# Data Mining 2 — Modélisation Statistique et Apprentissage


**YEMELO Bruel**

---

# Mini-projet Machine Learning — Classification bancaire

## Sujet

On étudie le jeu de données **Bank Marketing** de l'UCI Machine Learning Repository. Les données proviennent de campagnes de marketing direct d'une institution bancaire portugaise. L'objectif est de prédire si un client va souscrire un dépôt à terme, c'est-à-dire prédire la variable cible `y` qui vaut `yes` ou `no`.

Dans ce notebook, on compare plusieurs algorithmes de classification :

- Naive Bayes ;
- CART ;
- Random Forest ;
- SVM linéaire ;
- SVM RBF ;
- réseau de neurones ;
- régression logistique Ridge.

On ajoute aussi une partie optionnelle sur les règles d'association.

Source : UCI Machine Learning Repository, Bank Marketing Dataset.


## Questions générales

1. Quelle est la variable cible ? Est-elle binaire ou multiclasse ?
2. Quelles variables sont numériques ? Quelles variables sont catégorielles ?
3. Le jeu de données est-il équilibré ? Pourquoi l'accuracy seule peut-elle être trompeuse ?
4. Pourquoi faut-il encoder les variables catégorielles avant d'entraîner un SVM ou un réseau de neurones ?
5. Pourquoi faut-il standardiser les variables numériques pour les SVM, les réseaux de neurones et la régression Ridge ?
6. Quel modèle obtient le meilleur compromis entre précision, rappel, F1-score et AUC ?
7. Les différences entre les modèles sont-elles statistiquement significatives ?


## 1. Importation des bibliothèques

Exécutez la cellule suivante. Si une bibliothèque manque, installez-la dans votre environnement Python.


In [ ]:
#conda install numpy scipy matplotlib scikit-learn pandas -y

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import zipfile
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import friedmanchisquare, wilcoxon

from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    RocCurveDisplay
)

from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression


## 2. Chargement du jeu de données

Pour garder un temps de calcul raisonnable, on utilise par défaut `bank-additional.csv`, qui contient environ 10 % du jeu complet. Pour utiliser le jeu complet, remplacez `bank-additional.csv` par `bank-additional-full.csv` dans la cellule.


In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank-additional.zip"
zip_path = "bank-additional.zip"
data_dir = "bank-additional"

if not os.path.exists(zip_path):
    urllib.request.urlretrieve(url, zip_path)

if not os.path.exists(data_dir):
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(".")

# Version légère pour faciliter l'exécution du notebook.
# Pour le jeu complet, utiliser : data_file = "bank-additional/bank-additional-full.csv"
data_file = "bank-additional/bank-additional.csv"

df = pd.read_csv(data_file, sep=";")

df.head()


In [ ]:
print("Dimensions du jeu de données :", df.shape)
print("\nColonnes :")
print(df.columns.tolist())
print("\nAperçu des types :")
print(df.dtypes)


### Questions

1. Combien y a-t-il d'observations ?
2. Combien y a-t-il de variables explicatives ?
3. Quel est le type de la variable cible `y` ?
4. Quelles variables sont qualitatives ? Quelles variables sont quantitatives ?


### Réponses

**1. Combien y a-t-il d'observations ?**

Le jeu de données Bank Marketing contient **4119 observations**.

---

**2. Combien y a-t-il de variables explicatives ?**

Le jeu de données contient **21 variables** au total. Parmi elles, **20 sont des variables explicatives** et **1 variable correspond à la variable cible (`y`)**.

---

**3. Quel est le type de la variable cible `y` ?**

La variable cible `y` indique si un client a souscrit (`yes`) ou non (`no`) à un dépôt à terme proposé par la banque. Elle est donc une **variable qualitative binaire**. Le problème étudié est ainsi un problème de **classification binaire**.

---

**4. Quelles variables sont qualitatives ? Quelles variables sont quantitatives ?**

Les variables **quantitatives** sont :

- `age`
- `duration`
- `campaign`
- `pdays`
- `previous`
- `emp.var.rate`
- `cons.price.idx`
- `cons.conf.idx`
- `euribor3m`
- `nr.employed`


Les variables **qualitatives** sont :

- `job`
- `marital`
- `education`
- `default`
- `housing`
- `loan`
- `contact`
- `month`
- `day_of_week`
- `poutcome`



## 3. Analyse exploratoire


In [ ]:
# Distribution de la variable cible
class_counts = df["y"].value_counts()
class_props = df["y"].value_counts(normalize=True)

print(class_counts)
print("\nProportions :")
print(class_props)

plt.figure(figsize=(5, 4))
class_counts.plot(kind="bar")
plt.title("Distribution de la variable cible y")
plt.xlabel("Classe")
plt.ylabel("Effectif")
plt.xticks(rotation=0)
plt.show()


In [ ]:
# Statistiques descriptives des variables numériques
num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = df.select_dtypes(include=["object"]).drop(columns=["y"]).columns.tolist()

print("Variables numériques :", num_cols)
print("Variables catégorielles :", cat_cols)

df[num_cols].describe().T


In [ ]:
# Taux de souscription selon quelques variables catégorielles
for col in ["job", "marital", "education", "contact", "month"]:
    tab = pd.crosstab(df[col], df["y"], normalize="index").sort_values("yes", ascending=False)
    print("\n", col)
    display(tab.head(10))


### Questions

1. La classe `yes` est-elle minoritaire ?
2. Quelles variables semblent liées à la souscription ?
3. Pourquoi le rappel de la classe `yes` est-il important dans ce problème ?
4. Que signifie une valeur élevée de l'AUC ?


### Réponses

**1. La classe `yes` est-elle minoritaire ?**

Oui. La classe `yes` est  minoritaire. D'après la distribution observée, environ **11 % des clients ont souscrit** à un dépôt à terme contre près de **89 % qui n'ont pas souscrit**. Le jeu de données est donc déséquilibré.

---

**2. Quelles variables semblent liées à la souscription ?**

L'analyse exploratoire met en évidence plusieurs variables associées à la souscription :

- **Job** : les étudiants et les retraités présentent des taux de souscription relativement élevés (environ 23 %), tandis que les employés des services ou les indépendants présentent des taux plus faibles.
- **Situation matrimoniale** : les personnes célibataires semblent souscrire davantage que les personnes mariées ou divorcées.
- **Niveau d'éducation** : les personnes ayant un diplôme universitaire ou une formation professionnelle présentent des taux de souscription plus élevés que celles ayant uniquement une éducation de base.
- **Mode de contact** : les clients contactés par téléphone portable (*cellular*) souscrivent davantage que ceux contactés par téléphone fixe.
- **Mois du contact** : les taux de souscription varient fortement selon le mois. Les mois de mars, décembre, septembre et octobre affichent des proportions de souscription nettement plus élevées que mai, juin ou juillet.

Ces différences suggèrent que ces variables pourraient jouer un rôle important dans les modèles de classification.

---

**3. Pourquoi le rappel de la classe `yes` est-il important dans ce problème ?**

Le rappel (*recall*) mesure la proportion de clients ayant réellement souscrit qui sont correctement identifiés par le modèle.

Dans ce contexte bancaire, un faux négatif correspond à un client susceptible de souscrire mais que le modèle ne détecte pas. Cela représente une opportunité commerciale perdue pour la banque. Il est donc important d'obtenir un rappel élevé afin d'identifier le plus grand nombre possible de clients potentiellement intéressés par le produit proposé.

---

**4. Que signifie une valeur élevée de l'AUC ?**

L'AUC (*Area Under the ROC Curve*) mesure la capacité du modèle à distinguer les clients qui souscrivent de ceux qui ne souscrivent pas.

Une valeur élevée de l'AUC indique que le modèle attribue généralement des scores plus élevés aux clients qui souscrivent qu'à ceux qui ne souscrivent pas. Plus l'AUC est proche de 1, meilleure est la capacité de discrimination du modèle. À l'inverse, une AUC proche de 0,5 correspond à des performances comparables à un classement aléatoire.

## 4. Prétraitement

On sépare la variable cible et les variables explicatives. La cible `yes/no` est transformée en `1/0`.

On construit ensuite deux prétraitements :

- standardisation des variables numériques ;
- encodage one-hot des variables catégorielles.


In [ ]:
X = df.drop(columns=["y"])
y = df["y"].map({"no": 0, "yes": 1})

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Variables numériques :", numeric_features)
print("Variables catégorielles :", categorical_features)

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Train :", X_train.shape)
print("Test :", X_test.shape)
print("Proportion de yes dans train :", y_train.mean())
print("Proportion de yes dans test :", y_test.mean())


### Rappel : prétraitement, Pipeline et fuite de données

Avant d'entraîner un modèle, il est souvent nécessaire de transformer les
données. Par exemple, on peut :

- remplacer les valeurs manquantes ;
- standardiser les variables numériques ;
- encoder les variables catégorielles ;
- sélectionner certaines variables.

Ces opérations semblent simples, mais elles doivent être faites avec
prudence. En effet, certaines transformations apprennent des informations à
partir des données. Par exemple, une standardisation calcule une moyenne et
un écart-type :

$$
z = \frac{x-\mu}{\sigma}.
$$

Si $\mu$ et $\sigma$ sont calculés sur l'ensemble complet des données
avant la séparation apprentissage/test, alors le jeu de test influence déjà
le prétraitement. On parle alors de fuite de données, ou *data leakage*.

Le jeu de test doit représenter des données nouvelles, inconnues au moment
de l'apprentissage. Il ne doit donc pas être utilisé pour calculer les
paramètres du prétraitement.

Un `Pipeline` permet d'éviter ce problème. Il regroupe dans un même objet
les étapes de prétraitement et le modèle :

$$
\text{prétraitement} \longrightarrow \text{modèle}.
$$

Ainsi, lorsqu'on entraîne le pipeline sur les données d'apprentissage, les
paramètres du prétraitement sont calculés uniquement sur ces données. Ensuite,
les mêmes transformations sont appliquées au jeu de test, sans recalculer les
paramètres sur celui-ci.

Cela est particulièrement important en validation croisée, car le pipeline
garantit que, pour chaque découpage, les transformations sont apprises
uniquement sur la partie d'entraînement du pli considéré.

### Question théorique

Pourquoi doit-on appliquer le prétraitement à l'intérieur d'un `Pipeline` plutôt que de transformer toutes les données avant la séparation apprentissage/test ?


### Réponse

Le prétraitement doit être appliqué à l'intérieur d'un `Pipeline` afin d'éviter les problèmes de **fuite de données (data leakage)**. En effet, si les transformations (standardisation, encodage, normalisation, etc.) sont réalisées sur l'ensemble des données avant la séparation apprentissage/test, les informations contenues dans le jeu de test sont utilisées pour calculer les paramètres du prétraitement.

Or, le jeu de test est censé représenter de nouvelles observations inconnues au moment de l'apprentissage. Il ne doit donc pas influencer la construction du modèle ni les étapes de prétraitement.

L'utilisation d'un `Pipeline` garantit que les paramètres du prétraitement sont estimés uniquement à partir des données d'apprentissage, puis appliqués au jeu de test sans être recalculés. Cette procédure permet d'obtenir une évaluation plus fiable des performances du modèle.


## 4 bis. Naive Bayes selon la nature des variables

Dans le prétraitement général du projet, les variables catégorielles sont encodées par **one-hot encoding**. Cette représentation est adaptée à de nombreux modèles comme les SVM, les réseaux de neurones ou la régression logistique Ridge.

Pour Naive Bayes, il est utile de distinguer plusieurs variantes selon la nature des variables utilisées :

- **Gaussian Naive Bayes** sur les variables numériques : on suppose que, conditionnellement à la classe, chaque variable numérique suit une loi normale ;
- **Categorical Naive Bayes** sur les variables catégorielles : les modalités sont représentées par des entiers, uniquement comme codage technique ;
- **Bernoulli Naive Bayes** sur les variables catégorielles encodées en one-hot : chaque modalité devient une variable binaire.

L'objectif est de comparer ces variantes sur le **même jeu de données bancaire**, sans introduire un autre dataset.

### Rappel théorique

Le modèle Naive Bayes repose sur l'hypothèse d'indépendance conditionnelle :

$$
P(X_1=x_1,\ldots,X_p=x_p \mid Y=c)
=
\prod_{j=1}^p P(X_j=x_j \mid Y=c).
$$

Pour les variables numériques, Gaussian Naive Bayes suppose :

$$
X_j \mid Y=c \sim \mathcal{N}(\mu_{jc},\sigma_{jc}^2).
$$

Pour les variables catégorielles, on estime des probabilités conditionnelles de modalités. Si l'on utilise le one-hot encoding, les variables obtenues sont binaires, ce qui justifie l'utilisation de Bernoulli Naive Bayes.

### Question

Pourquoi n'utilise-t-on pas exactement le même prétraitement pour `GaussianNB`, `CategoricalNB` et `BernoulliNB` ? Reliez votre réponse à la nature des variables et aux hypothèses probabilistes de chaque modèle.


### Réponse

On n'utilise pas exactement le même prétraitement pour `GaussianNB`, `CategoricalNB` et `BernoulliNB` car chacun de ces modèles repose sur des hypothèses probabilistes différentes concernant la nature des variables explicatives.

`GaussianNB` est adapté aux variables quantitatives. Il suppose que, pour chaque classe, les variables suivent une distribution normale (gaussienne). Les variables numériques peuvent donc être utilisées directement, éventuellement après standardisation.

`CategoricalNB` est conçu pour des variables catégorielles prenant un nombre fini de modalités. Les catégories doivent être converties en codes numériques représentant les différentes modalités, mais sans créer de variables binaires.

`BernoulliNB` est quant à lui adapté aux variables binaires. Après un encodage de type *one-hot encoding*, chaque modalité est représentée par une variable prenant uniquement les valeurs 0 ou 1. Cette représentation correspond directement à l'hypothèse du modèle Bernoulli.

Ainsi, le choix du prétraitement dépend de la nature des variables et des hypothèses probabilistes formulées par chaque variante de Naive Bayes. Utiliser un prétraitement inadapté pourrait violer ces hypothèses et dégrader les performances du modèle.

In [ ]:
from sklearn.naive_bayes import GaussianNB, CategoricalNB, BernoulliNB
from sklearn.preprocessing import OrdinalEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer

# Fonction d'évaluation commune aux variantes de Naive Bayes

def evaluate_nb_variant(name, pipe):
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_score = pipe.predict_proba(X_test)[:, 1]
    
    return {
        "Modèle": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "AUC": roc_auc_score(y_test, y_score),
        "y_pred": y_pred,
        "y_score": y_score,
        "pipe": pipe
    }

# ---------------------------------------------------------------------
# 1. Gaussian Naive Bayes : variables numériques uniquement
# ---------------------------------------------------------------------

numeric_preprocess_nb = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_features)
    ]
)

gnb_numeric_pipeline = Pipeline(steps=[
    ("preprocess", numeric_preprocess_nb),
    ("model", GaussianNB())
])

# ---------------------------------------------------------------------
# 2. Categorical Naive Bayes : variables catégorielles avec encodage ordinal
# ---------------------------------------------------------------------
# L'encodage ordinal est ici un codage technique attendu par CategoricalNB.
# Pour éviter les valeurs négatives en cas de modalité inconnue, on ajoute 1
# après l'encodage. Les catégories vues à l'entraînement sont donc codées
# 1, 2, 3, ... et une éventuelle catégorie inconnue serait codée 0.

categorical_preprocess_cnb = ColumnTransformer(
    transformers=[
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1
            )),
            ("shift", FunctionTransformer(lambda X: X + 1, feature_names_out="one-to-one"))
        ]), categorical_features)
    ]
)

cnb_categorical_pipeline = Pipeline(steps=[
    ("preprocess", categorical_preprocess_cnb),
    ("model", CategoricalNB(alpha=1.0))
])

# ---------------------------------------------------------------------
# 3. Bernoulli Naive Bayes : variables catégorielles avec one-hot encoding
# ---------------------------------------------------------------------

categorical_preprocess_bnb = ColumnTransformer(
    transformers=[
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features)
    ]
)

bnb_onehot_pipeline = Pipeline(steps=[
    ("preprocess", categorical_preprocess_bnb),
    ("model", BernoulliNB(alpha=1.0))
])

# Évaluation des trois variantes
nb_variant_results = []

for name, pipe in {
    "Naive Bayes gaussien (variables numériques)": gnb_numeric_pipeline,
    "Naive Bayes catégoriel (encodage ordinal)": cnb_categorical_pipeline,
    "Naive Bayes Bernoulli (one-hot)": bnb_onehot_pipeline
}.items():
    print("Entraînement :", name)
    nb_variant_results.append(evaluate_nb_variant(name, pipe))

nb_variants_df = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ["y_pred", "y_score", "pipe"]}
    for r in nb_variant_results
])

nb_variants_df.sort_values("F1", ascending=False)


### Questions : comparaison des variantes de Naive Bayes

1. Quelles variables sont utilisées par le Naive Bayes gaussien ? Quelles variables sont utilisées par les deux variantes catégorielles ?
2. Quelle hypothèse de distribution est faite par Gaussian Naive Bayes ? Cette hypothèse vous semble-t-elle réaliste pour des variables comme `age`, `duration` ou `campaign` ?
3. Pourquoi `CategoricalNB` utilise-t-il un encodage ordinal alors que les autres modèles du projet utilisent plutôt le one-hot encoding ?
4. Pourquoi `BernoulliNB` est-il cohérent avec des variables issues d'un one-hot encoding ?
5. Comparez les performances des trois variantes. Les variables numériques seules suffisent-elles à obtenir une bonne performance ?
6. Quelle variante de Naive Bayes recommanderiez-vous sur ce jeu de données ? Justifiez en tenant compte des métriques et des hypothèses du modèle.

### Remarque

Un modèle Naive Bayes mixte pourrait théoriquement combiner les deux types de variables :

$$
P(X \mid Y=c)
=
\prod_{j \in \mathcal{N}} p(x_j \mid Y=c)
\prod_{k \in \mathcal{C}} P(x_k \mid Y=c),
$$

où $\mathcal{N}$ désigne les variables numériques et $\mathcal{C}$ les variables catégorielles. Dans ce notebook, on compare séparément les variantes afin de bien identifier le rôle de chaque type de variable et de chaque hypothèse probabiliste.


### Réponses : comparaison des variantes de Naive Bayes

**1. Quelles variables sont utilisées par le Naive Bayes gaussien ? Quelles variables sont utilisées par les deux variantes catégorielles ?**

Le Naive Bayes gaussien utilise uniquement les variables numériques. Dans le pipeline, ces variables sont imputées par la médiane puis standardisées avant l'entraînement du modèle `GaussianNB`.

Les deux variantes catégorielles utilisent uniquement les variables qualitatives. `CategoricalNB` utilise un encodage ordinal des modalités, tandis que `BernoulliNB` utilise un encodage one-hot, qui transforme chaque modalité en variable binaire.

---

**2. Quelle hypothèse de distribution est faite par Gaussian Naive Bayes ? Cette hypothèse vous semble-t-elle réaliste pour des variables comme `age`, `duration` ou `campaign` ?**

`GaussianNB` suppose que les variables numériques suivent une loi normale conditionnellement à la classe `y`. Autrement dit, pour chaque classe (`yes` ou `no`), chaque variable numérique est supposée distribuée selon une loi gaussienne.

Cette hypothèse peut être approximativement acceptable pour certaines variables comme `age`, mais elle est moins réaliste pour `duration` ou `campaign`, qui sont souvent asymétriques et peuvent contenir des valeurs extrêmes. Malgré cela, le modèle peut rester performant en pratique.

---

**3. Pourquoi `CategoricalNB` utilise-t-il un encodage ordinal alors que les autres modèles du projet utilisent plutôt le one-hot encoding ?**

`CategoricalNB` est conçu pour traiter des variables catégorielles discrètes codées sous forme d'entiers. L'encodage ordinal sert donc ici de codage technique : chaque modalité reçoit un numéro.

Dans le code, les modalités inconnues sont codées `-1`, puis on ajoute 1 avec `FunctionTransformer`, afin d'éviter les valeurs négatives. Les catégories connues sont donc codées 1, 2, 3, etc., et une éventuelle catégorie inconnue serait codée 0.

Les autres modèles utilisent plutôt le one-hot encoding pour éviter d'introduire artificiellement un ordre entre les catégories.

---

**4. Pourquoi `BernoulliNB` est-il cohérent avec des variables issues d'un one-hot encoding ?**

`BernoulliNB` est adapté aux variables binaires, c'est-à-dire aux variables qui valent 0 ou 1. Or, après un one-hot encoding, chaque modalité devient une variable binaire indiquant la présence ou l'absence de cette modalité. Cette représentation est donc cohérente avec les hypothèses de `BernoulliNB`.

---

**5. Comparez les performances des trois variantes. Les variables numériques seules suffisent-elles à obtenir une bonne performance ?**

Le Naive Bayes gaussien obtient le meilleur F1-score et la meilleure AUC. Il obtient aussi le meilleur rappel, ce qui signifie qu'il détecte mieux les clients ayant effectivement souscrit. Le Naive Bayes catégoriel obtient une accuracy et une précision plus élevées, mais son rappel est faible. Il identifie donc moins bien la classe `yes`.

Les variables numériques seules semblent donc contenir une information importante pour prédire la souscription, notamment grâce à des variables comme `duration`, `campaign`, `pdays`, `previous` et les variables économiques.

---

**6. Quelle variante de Naive Bayes recommanderiez-vous sur ce jeu de données ?**

Je recommanderais le Naive Bayes gaussien. Même si son accuracy n'est pas la plus élevée, il présente le meilleur compromis global selon le F1-score, le rappel et l'AUC. Dans ce problème bancaire, il est important de bien identifier les clients susceptibles de souscrire ; le rappel et le F1-score sont donc plus pertinents que la seule accuracy.

## 5. Définition des modèles

Les modèles étudiés sont :

- **Naive Bayes** : modèle probabiliste simple ;
- **CART** : arbre de décision ;
- **Random Forest** : ensemble d'arbres par bagging ;
- **SVM linéaire** : séparation linéaire avec marge ;
- **SVM RBF** : séparation non linéaire par noyau gaussien ;
- **Réseau de neurones** : perceptron multicouche ;
- **Ridge Logistic Regression** : régression logistique pénalisée en norme \(L_2\).


### Rappel : SVM, noyau, déséquilibre de classes et frontière de décision

Un SVM cherche une frontière de décision qui maximise la marge entre les
classes. Dans le cas linéaire, cette frontière est une droite ou un hyperplan.
Dans le cas non linéaire, on peut utiliser un noyau, par exemple le noyau RBF,
qui permet d'obtenir une frontière de décision plus flexible.

Le paramètre $C$ contrôle le compromis entre marge large et erreurs de
classification. Un petit $C$ autorise plus d'erreurs mais donne souvent une
frontière plus régulière. Un grand $C$ pénalise fortement les erreurs et
peut conduire à une frontière plus complexe.

Le paramètre $\gamma$, utilisé avec le noyau RBF, contrôle l'influence locale
des observations. Une petite valeur de $\gamma$ donne une frontière plus
lisse, tandis qu'une grande valeur de $\gamma$ peut donner une frontière
très irrégulière.

Lorsque les données sont déséquilibrées, le SVM peut favoriser la classe
majoritaire. Dans ce cas, la frontière de décision peut se déplacer de manière
à mieux classer la classe majoritaire, au détriment de la classe minoritaire.
L'option `class_weight="balanced"` augmente automatiquement le poids des
observations de la classe minoritaire afin de pénaliser davantage leurs erreurs
de classification.

On peut donc comparer deux situations :

$$
\text{SVM non pondéré}
$$

et

$$
\text{SVM pondéré avec class\_weight="balanced"}.
$$

L'objectif est d'étudier si la pondération améliore le rappel, le F1-score ou
l'AUC pour la classe minoritaire.  


In [ ]:
models = {
    "Naive Bayes": GaussianNB(),
    "CART": DecisionTreeClassifier(max_depth=5, random_state=42, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=None, random_state=42, class_weight="balanced", n_jobs=-1
    ),
    "SVM linéaire balanced": SVC(kernel="linear", C=1.0, probability=True, class_weight="balanced", random_state=42),
    "SVM linéaire non pondéré": SVC(kernel="linear", C=1.0, probability=True, class_weight=None, random_state=42),
    "SVM RBF": SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, class_weight=None, random_state=42),
    "SVM RBF balanced": SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, class_weight="balanced", random_state=42),
    "Neural Network": MLPClassifier(
        hidden_layer_sizes=(32, 16), activation="relu", alpha=1e-4,
        max_iter=500, random_state=42, early_stopping=True
    ),
    "Ridge Logistic": LogisticRegression(
        penalty="l2", C=1.0, solver="lbfgs", max_iter=2000, class_weight="balanced"
    )
}

pipelines = {
    name: Pipeline(steps=[("preprocess", preprocess), ("model", model)])
    for name, model in models.items()
}


### Questions

1. Pourquoi ne doit-on pas choisir arbitrairement le noyau du SVM ?

2. Quel est le rôle du paramètre $C$ dans un SVM ?

3. Quel est le rôle du paramètre $\gamma$ dans un SVM avec noyau RBF ?

4. Pourquoi le déséquilibre des classes peut-il influencer la frontière de
   décision d'un SVM ?

5. Que fait l'option `class_weight="balanced"` ?

6. Comparez les résultats obtenus avec `class_weight=None` et
   `class_weight="balanced"`. Quelle configuration donne le meilleur rappel
   pour la classe minoritaire ?

7. La meilleure accuracy correspond-elle nécessairement au meilleur modèle
   dans un problème déséquilibré ? Justifiez.

### Réponses aux questions sur les SVM

**1. Pourquoi ne doit-on pas choisir arbitrairement le noyau du SVM ?**

Le noyau détermine la forme de la frontière de décision apprise par le modèle. Un noyau linéaire est adapté lorsque les classes sont approximativement séparables par un hyperplan, tandis qu'un noyau non linéaire comme le noyau RBF permet de capturer des relations plus complexes entre les variables. Le choix du noyau influence donc directement les performances du modèle et doit être validé empiriquement à l'aide des données.

---

**2. Quel est le rôle du paramètre \(C\) dans un SVM ?**

Le paramètre \(C\) contrôle le compromis entre la maximisation de la marge et la pénalisation des erreurs de classification. Une valeur élevée de \(C\) cherche à réduire les erreurs d'entraînement, au risque de produire une frontière de décision plus complexe et de sur-apprendre les données. À l'inverse, une valeur faible autorise davantage d'erreurs mais favorise une meilleure généralisation grâce à une marge plus large.

---

**3. Quel est le rôle du paramètre \(\gamma\) dans un SVM avec noyau RBF ?**

Le paramètre \(\gamma\) détermine l'influence d'une observation sur la frontière de décision. Une valeur élevée de \(\gamma\) rend le modèle très sensible aux observations proches et conduit à une frontière complexe pouvant sur-ajuster les données. Une valeur faible produit une frontière plus lisse et plus générale.

---

**4. Pourquoi le déséquilibre des classes peut-il influencer la frontière de décision d'un SVM ?**

Lorsque les classes sont déséquilibrées, la classe majoritaire exerce une influence plus importante sur l'apprentissage. Le SVM peut alors privilégier une bonne classification de cette classe au détriment de la classe minoritaire. Dans ce cas, le modèle peut obtenir une bonne accuracy globale tout en détectant mal les observations appartenant à la classe minoritaire.

---

**5. Que fait l'option `class_weight="balanced"` ?**

L'option `class_weight="balanced"` ajuste automatiquement les poids attribués aux classes en fonction de leur fréquence dans les données. Les erreurs commises sur la classe minoritaire sont davantage pénalisées que celles commises sur la classe majoritaire. Cette pondération permet généralement d'améliorer la détection de la classe minoritaire et d'obtenir un meilleur rappel pour cette classe.

## 6. Entraînement et évaluation sur l'ensemble de test


In [ ]:
def evaluate_model(name, pipe, X_train, X_test, y_train, y_test):
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    if hasattr(pipe[-1], "predict_proba"):
        y_score = pipe.predict_proba(X_test)[:, 1]
    else:
        y_score = pipe.decision_function(X_test)

    return {
        "Modèle": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "AUC": roc_auc_score(y_test, y_score),
        "y_pred": y_pred,
        "y_score": y_score,
        "pipe": pipe
    }

results = []
for name, pipe in pipelines.items():
    print("Entraînement :", name)
    results.append(evaluate_model(name, pipe, X_train, X_test, y_train, y_test))

metrics_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ["y_pred", "y_score", "pipe"]} for r in results])
metrics_df.sort_values("F1", ascending=False)


test_predictions = {
    r["Modèle"]: r["y_pred"]
    for r in results
}

test_scores = {
    r["Modèle"]: r["y_score"]
    for r in results
}

trained_pipelines = {
    r["Modèle"]: r["pipe"]
    for r in results
}


In [ ]:
metrics_df[metrics_df["Modèle"].str.contains("SVM")].sort_values("Recall", ascending=False)

In [ ]:
metrics_df.loc[
    metrics_df["Modèle"].isin([
        "SVM linéaire balanced",
        "SVM linéaire non pondéré",
        "SVM RBF",
        "SVM RBF balanced"
    ]),
    ["Modèle", "Accuracy", "Precision", "Recall", "F1", "AUC"]
].sort_values("Recall", ascending=False)

### Réponse question 6 SVM

**6. Comparez les résultats obtenus avec `class_weight=None` et `class_weight="balanced"`. Quelle configuration donne le meilleur rappel pour la classe minoritaire ?**

L'utilisation de `class_weight="balanced"` améliore fortement le rappel de la classe minoritaire (`yes`). Pour le SVM linéaire, le rappel passe d'environ 0,354 à 0,858 lorsque l'on active la pondération des classes. Pour le SVM à noyau RBF, le rappel passe d'environ 0,248 à 0,832.

Cette amélioration s'explique par le fait que les erreurs commises sur la classe minoritaire sont davantage pénalisées lors de l'apprentissage. Le modèle accorde donc plus d'importance aux observations de la classe `yes` et parvient à en identifier davantage.

Parmi les quatre configurations testées, le **SVM linéaire avec `class_weight="balanced"`** obtient le meilleur rappel pour la classe minoritaire (0,858).

## 6 bis. Intervalles de confiance par bootstrap

Les métriques calculées sur le jeu de test sont des estimations obtenues sur un seul échantillon. Elles peuvent donc varier si l'on change l'échantillon de test. Pour quantifier cette incertitude, on peut construire des **intervalles de confiance bootstrap**.

Le principe est le suivant :

1. on tire avec remise des observations du jeu de test ;
2. on recalcule la métrique sur cet échantillon bootstrap ;
3. on répète l'opération un grand nombre de fois ;
4. on prend les quantiles 2.5 % et 97.5 % de la distribution obtenue.

On obtient ainsi un intervalle de confiance approximatif à 95 %.

### Questions

1. Pourquoi les scores mesurés sur le jeu de test sont-ils soumis à de l'incertitude ?
2. Pourquoi le bootstrap tire-t-il les observations **avec remise** ?
3. Pourquoi un intervalle de confiance large indique-t-il une estimation moins précise ?
4. Deux modèles dont les intervalles de confiance se recouvrent fortement sont-ils nécessairement différents statistiquement ?
5. Pourquoi faut-il compléter cette analyse par un test apparié comme le test de McNemar ?


### Réponses aux questions

**1. Pourquoi les scores mesurés sur le jeu de test sont-ils soumis à de l'incertitude ?**

Les métriques calculées sur le jeu de test (accuracy, rappel, F1-score, AUC, etc.) sont obtenues sur un seul échantillon de données. Si l'on changeait le découpage apprentissage/test ou si l'on observait un autre échantillon de la population, les valeurs obtenues pourraient être différentes. Les scores observés sont donc des estimations soumises à une variabilité d'échantillonnage.

---

**2. Pourquoi le bootstrap tire-t-il les observations avec remise ?**

Le bootstrap cherche à reproduire la variabilité que l'on observerait si l'on disposait de plusieurs échantillons issus de la même population. Le tirage avec remise permet de générer de nouveaux échantillons bootstrap de même taille que l'échantillon initial tout en introduisant une variabilité artificielle. Certaines observations peuvent apparaître plusieurs fois tandis que d'autres peuvent ne pas être sélectionnées.

---

**3. Pourquoi un intervalle de confiance large indique-t-il une estimation moins précise ?**

Un intervalle de confiance large signifie que la métrique estimée varie fortement d'un échantillon bootstrap à l'autre. Cette forte variabilité traduit une plus grande incertitude sur la valeur réelle de la métrique. À l'inverse, un intervalle étroit indique une estimation plus stable et donc plus précise.

---

**4. Deux modèles dont les intervalles de confiance se recouvrent fortement sont-ils nécessairement différents statistiquement ?**

Non. Si les intervalles de confiance se recouvrent fortement, cela suggère que les différences observées entre les performances des modèles peuvent être dues au hasard de l'échantillonnage. Dans ce cas, on ne peut pas conclure à une différence statistiquement significative sur la seule base des intervalles de confiance.

---

**5. Pourquoi faut-il compléter cette analyse par un test apparié comme le test de McNemar ?**

Les intervalles de confiance décrivent l'incertitude associée aux performances individuelles des modèles, mais ils ne testent pas directement si deux modèles commettent des erreurs différentes sur les mêmes observations. Le test de McNemar compare précisément les prédictions de deux modèles sur un même jeu de test et permet d'évaluer si la différence observée entre leurs performances est statistiquement significative.

In [ ]:
def bootstrap_metric_ci(y_true, y_pred, y_score=None, metric="f1", n_bootstrap=1000, alpha=0.05, random_state=42):
    """
    Calcule un intervalle de confiance bootstrap pour une métrique de classification.

    Paramètres
    ----------
    y_true : array-like
        Vraies classes du jeu de test.
    y_pred : array-like
        Classes prédites par le modèle.
    y_score : array-like ou None
        Scores ou probabilités pour la classe positive. Nécessaire pour AUC.
    metric : str
        Une métrique parmi : "accuracy", "precision", "recall", "f1", "auc".
    n_bootstrap : int
        Nombre de ré-échantillonnages bootstrap.
    alpha : float
        Niveau d'erreur. alpha=0.05 donne un intervalle à 95 %.
    random_state : int
        Graine aléatoire.
    """
    rng = np.random.default_rng(random_state)

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    if y_score is not None:
        y_score = np.asarray(y_score)

    n = len(y_true)
    values = []

    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)
        yt = y_true[idx]
        yp = y_pred[idx]

        if metric == "accuracy":
            value = accuracy_score(yt, yp)
        elif metric == "precision":
            value = precision_score(yt, yp, zero_division=0)
        elif metric == "recall":
            value = recall_score(yt, yp, zero_division=0)
        elif metric == "f1":
            value = f1_score(yt, yp, zero_division=0)
        elif metric == "auc":
            # L'AUC n'est définie que si les deux classes sont présentes
            # dans l'échantillon bootstrap.
            if y_score is None or len(np.unique(yt)) < 2:
                continue
            value = roc_auc_score(yt, y_score[idx])
        else:
            raise ValueError("metric doit être parmi : accuracy, precision, recall, f1, auc")

        values.append(value)

    values = np.asarray(values)
    estimate = values.mean()
    lower = np.quantile(values, alpha / 2)
    upper = np.quantile(values, 1 - alpha / 2)

    return estimate, lower, upper, values


# Calcul des intervalles de confiance bootstrap pour chaque modèle.
# On réduit n_bootstrap si le notebook est trop lent.
bootstrap_rows = []
metrics_to_bootstrap = ["accuracy", "precision", "recall", "f1", "auc"]

for model_name in test_predictions.keys():
    y_pred_model = test_predictions[model_name]
    y_score_model = test_scores.get(model_name, None)

    for metric in metrics_to_bootstrap:
        estimate, lower, upper, _ = bootstrap_metric_ci(
            y_test,
            y_pred_model,
            y_score=y_score_model,
            metric=metric,
            n_bootstrap=1000,
            alpha=0.05,
            random_state=42
        )

        bootstrap_rows.append({
            "Modèle": model_name,
            "Métrique": metric,
            "Estimation bootstrap": estimate,
            "IC 95% bas": lower,
            "IC 95% haut": upper,
            "Largeur IC": upper - lower
        })

bootstrap_ci_df = pd.DataFrame(bootstrap_rows)
bootstrap_ci_df.sort_values(["Métrique", "Estimation bootstrap"], ascending=[True, False])


In [ ]:
# Visualisation des intervalles de confiance pour le F1-score
ci_f1 = bootstrap_ci_df[bootstrap_ci_df["Métrique"] == "f1"].copy()
ci_f1 = ci_f1.sort_values("Estimation bootstrap", ascending=True)

plt.figure(figsize=(8, 5))
plt.errorbar(
    ci_f1["Estimation bootstrap"],
    ci_f1["Modèle"],
    xerr=[
        ci_f1["Estimation bootstrap"] - ci_f1["IC 95% bas"],
        ci_f1["IC 95% haut"] - ci_f1["Estimation bootstrap"]
    ],
    fmt="o",
    capsize=4
)
plt.xlabel("F1-score avec intervalle de confiance bootstrap à 95 %")
plt.ylabel("Modèle")
plt.title("Incertitude des performances par bootstrap")
plt.grid(alpha=0.3)
plt.show()


### Interprétation attendue

Les intervalles de confiance bootstrap donnent une indication de la stabilité des performances mesurées sur le jeu de test. Un modèle avec un F1-score élevé mais un intervalle très large peut être moins fiable qu'un modèle légèrement moins performant mais plus stable.

Cependant, le recouvrement des intervalles de confiance ne constitue pas à lui seul un test statistique apparié entre deux classifieurs. Comme les modèles sont évalués sur les mêmes observations, le test de McNemar reste utile pour comparer directement leurs erreurs deux à deux.


In [ ]:
# Matrices de confusion et rapports de classification
for r in results:
    print("=" * 80)
    print(r["Modèle"])
    print("Matrice de confusion :")
    print(confusion_matrix(y_test, r["y_pred"]))
    print("\nRapport de classification :")
    print(classification_report(y_test, r["y_pred"], target_names=["no", "yes"], zero_division=0))


In [ ]:
# Courbes ROC
plt.figure(figsize=(8, 6))
ax = plt.gca()
for r in results:
    RocCurveDisplay.from_predictions(y_test, r["y_score"], name=r["Modèle"], ax=ax)
plt.title("Courbes ROC des modèles")
plt.show()


### Questions

1. Quel modèle donne la meilleure accuracy ?
2. Quel modèle donne le meilleur rappel pour la classe `yes` ?
3. Quel modèle donne le meilleur F1-score ?
4. Quel modèle donne la meilleure AUC ?
5. Quel modèle choisiriez-vous pour une campagne bancaire réelle ? Justifiez.


### Réponses aux questions

**1. Quel modèle donne la meilleure accuracy ?**

Le SVM linéaire non pondéré obtient la meilleure accuracy, avec environ 91 % de bonnes classifications. Cela signifie qu'il est celui qui commet le moins d'erreurs sur l'ensemble du jeu de test.

**2. Quel modèle donne le meilleur rappel pour la classe `yes` ?**

Le SVM linéaire balanced présente le meilleur rappel pour la classe `yes` (0,86). Il parvient ainsi à identifier près de 86 % des clients ayant effectivement souscrit à l'offre, ce qui constitue une performance importante dans un contexte de marketing bancaire.

**3. Quel modèle donne le meilleur F1-score ?**

Le CART, le SVM linéaire balanced, le SVM RBF balanced et la régression logistique Ridge obtiennent tous un F1-score d'environ 0,57 pour la classe `yes`. Ces modèles offrent donc le meilleur compromis entre la précision et le rappel.

**4. Quel modèle donne la meilleure AUC ?**

Le Random Forest obtient la meilleure AUC, proche de 0,94. Cette métrique mesure la capacité du modèle à distinguer les clients susceptibles de souscrire de ceux qui ne souscriront pas. Une AUC aussi élevée indique une très bonne capacité de discrimination.

**5. Quel modèle choisiriez-vous pour une campagne bancaire réelle ? Justifiez.**

Pour une campagne bancaire, l'objectif principal est de repérer le plus grand nombre possible de clients susceptibles de répondre positivement à l'offre. Dans ce contexte, le rappel de la classe `yes` est particulièrement important. Le SVM linéaire balanced est donc le choix le plus pertinent. Même si son accuracy est légèrement inférieure à celle de certains autres modèles, il détecte la plus grande proportion de clients intéressés tout en conservant un bon équilibre entre précision et rappel. Il permet ainsi de réduire le risque de passer à côté de clients potentiellement intéressés par l'offre proposée.

### Recherche en grille pour SVM

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Identifier les variables numériques et catégorielles
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X_train.select_dtypes(include=["object", "category", "bool"]).columns

# Prétraitement des variables numériques
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Prétraitement des variables catégorielles
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Préprocesseur global
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [ ]:
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    make_scorer,
    f1_score
)

# Vérifier les classes
print("Classes dans y_train :", sorted(y_train.unique()))

# Comme y_train contient 0 et 1, la classe positive est 1
f1_positive = make_scorer(f1_score, pos_label=1)

# Pipeline SVM
svm_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", SVC(probability=True, random_state=42))
])

# Grille de paramètres
svm_param_grid = [
    {
        "model__kernel": ["linear"],
        "model__C": [0.01, 0.1, 1, 10, 100],
        "model__class_weight": [None, "balanced"]
    },
    {
        "model__kernel": ["rbf"],
        "model__C": [0.1, 1, 10, 100],
        "model__gamma": ["scale", 0.01, 0.1, 1],
        "model__class_weight": [None, "balanced"]
    }
]

# Validation croisée stratifiée
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Recherche en grille
svm_grid = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=svm_param_grid,
    scoring=f1_positive,
    cv=cv,
    n_jobs=-1,
    verbose=1,
    return_train_score=True,
    error_score="raise"
)

# Entraînement
svm_grid.fit(X_train, y_train)

print("Meilleurs paramètres SVM :")
print(svm_grid.best_params_)

print("\nMeilleur score F1 en validation croisée :")
print(svm_grid.best_score_)

### Evaluation sur le test

In [ ]:
# Meilleur modèle SVM
best_svm = svm_grid.best_estimator_

# Prédictions
y_pred_svm = best_svm.predict(X_test)
y_proba_svm = best_svm.predict_proba(X_test)[:, 1]

print("Matrice de confusion :")
print(confusion_matrix(y_test, y_pred_svm))

print("\nRapport de classification :")
print(classification_report(y_test, y_pred_svm))

print("\nAUC ROC :")
print(roc_auc_score(y_test, y_proba_svm))


positive_index = list(best_svm.classes_).index(1)
y_score_svm = best_svm.predict_proba(X_test)[:, positive_index]

test_predictions["SVM optimisé"] = y_pred_svm
test_scores["SVM optimisé"] = y_score_svm
trained_pipelines["SVM optimisé"] = best_svm

### Comparer explicitement balanced vs non-balanced

In [ ]:
svm_comparison_param_grid = {
    "model__kernel": ["rbf"],
    "model__C": [0.1, 1, 10],
    "model__gamma": ["scale", 0.01, 0.1],
    "model__class_weight": [None, "balanced"]
}

svm_comparison_grid = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=svm_comparison_param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

svm_comparison_grid.fit(X_train, y_train)

results_svm = pd.DataFrame(svm_comparison_grid.cv_results_)

cols = [
    "param_model__kernel",
    "param_model__C",
    "param_model__gamma",
    "param_model__class_weight",
    "mean_test_score",
    "std_test_score",
    "mean_train_score"
]

results_svm[cols].sort_values("mean_test_score", ascending=False).head(10)

### Comparaison par type de pondération

In [ ]:
summary_weight = (
    results_svm
    .groupby("param_model__class_weight")["mean_test_score"]
    .agg(["mean", "std", "max"])
    .reset_index()
)

summary_weight

# ou 

results_svm["weight_type"] = results_svm["param_model__class_weight"].astype(str)
results_svm["weight_type"] = results_svm["weight_type"].replace({
    "None": "non balanced",
    "balanced": "balanced"
})

results_svm.groupby("weight_type")["mean_test_score"].agg(["mean", "std", "max"])

### Question : déséquilibre de classes et frontière de décision

Dans un problème déséquilibré, la classe majoritaire contient beaucoup plus
d'observations que la classe minoritaire.

1. Que risque-t-il de se passer si on entraîne un SVM sans pondération des
   classes ?

2. Comment la frontière de décision peut-elle être influencée par la classe
   majoritaire ?

3. Pourquoi l'option `class_weight="balanced"` peut-elle déplacer la frontière
   de décision ?

4. Quel effet observe-t-on sur le rappel de la classe minoritaire ?

5. Est-il préférable d'optimiser l'accuracy, le F1-score ou l'AUC dans ce
   problème ? Justifiez.

### Réponses aux questions

**1. Que risque-t-il de se passer si on entraîne un SVM sans pondération des classes ?**

Lorsque les classes sont déséquilibrées, le SVM tend à privilégier la classe majoritaire. Il peut alors obtenir une bonne accuracy globale en classant correctement la majorité des observations, tout en détectant mal les observations appartenant à la classe minoritaire. Dans notre cas, le modèle risque donc de prédire trop souvent la classe `no` et de manquer de nombreux clients susceptibles de souscrire à l'offre.

**2. Comment la frontière de décision peut-elle être influencée par la classe majoritaire ?**

La classe majoritaire exerce une influence plus importante sur la position de la frontière de décision car elle contient davantage d'observations. Le SVM cherche alors à minimiser les erreurs sur cette classe dominante, ce qui peut déplacer la frontière vers la classe minoritaire et rendre sa détection plus difficile.

**3. Pourquoi l'option `class_weight="balanced"` peut-elle déplacer la frontière de décision ?**

L'option `class_weight="balanced"` attribue un poids plus élevé aux observations de la classe minoritaire. Les erreurs commises sur cette classe deviennent donc plus coûteuses pour le modèle. Le SVM ajuste alors sa frontière de décision afin de mieux prendre en compte les observations positives, ce qui améliore généralement leur détection.

**4. Quel effet observe-t-on sur le rappel de la classe minoritaire ?**

On observe une nette amélioration du rappel lorsque les classes sont pondérées. Dans nos résultats, les modèles utilisant `class_weight="balanced"` obtiennent des rappels beaucoup plus élevés que les modèles non pondérés. La pondération permet donc d'identifier beaucoup plus de clients appartenant à la classe minoritaire.

**5. Est-il préférable d'optimiser l'accuracy, le F1-score ou l'AUC dans ce problème ? Justifiez.**

Dans ce problème déséquilibré, l'accuracy n'est pas la métrique la plus pertinente car un modèle peut obtenir une accuracy élevée tout en détectant très mal la classe minoritaire. Le F1-score est généralement plus adapté puisqu'il tient compte à la fois de la précision et du rappel de la classe positive. L'AUC est également intéressante car elle mesure la capacité globale du modèle à distinguer les deux classes indépendamment du seuil de classification. Dans notre contexte, où l'objectif est de détecter les clients susceptibles de souscrire, le F1-score constitue le critère d'optimisation le plus pertinent.

## 7. Interprétation : importance des variables avec Random Forest


In [ ]:
rf_pipe = pipelines["Random Forest"]
rf_pipe.fit(X_train, y_train)

feature_names = rf_pipe.named_steps["preprocess"].get_feature_names_out()
importances = rf_pipe.named_steps["model"].feature_importances_

importance_df = pd.DataFrame({
    "variable": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

importance_df.head(20)


In [ ]:
plt.figure(figsize=(9, 6))
top = importance_df.head(20).sort_values("importance")
plt.barh(top["variable"], top["importance"])
plt.title("Top 20 des variables les plus importantes — Random Forest")
plt.xlabel("Importance")
plt.show()


### Questions

1. Quelles variables semblent les plus importantes ?
2. Ces variables étaient-elles déjà visibles dans l'analyse exploratoire ?
3. L'importance des variables d'une Random Forest implique-t-elle nécessairement une causalité ?


### Réponses aux questions

**1. Quelles variables semblent les plus importantes ?**

D'après les scores d'importance calculés par la Random Forest, la variable la plus importante est de loin **`duration`** (durée de l'appel téléphonique). Elle est suivie par plusieurs variables macroéconomiques telles que **`euribor3m`**, **`nr.employed`** et **`emp.var.rate`**, ainsi que par certaines caractéristiques individuelles comme **`age`**. Les résultats des campagnes précédentes (`poutcome_success`, `poutcome_nonexistent`) apparaissent également parmi les variables les plus influentes.

**2. Ces variables étaient-elles déjà visibles dans l'analyse exploratoire ?**

Oui, plusieurs de ces variables apparaissaient déjà comme importantes lors de l'analyse exploratoire. En particulier, la variable **`duration`** montrait une forte différence entre les clients ayant souscrit et ceux n'ayant pas souscrit. De même, certaines variables économiques et les informations relatives aux campagnes précédentes semblaient déjà associées au comportement de souscription. La Random Forest confirme donc en grande partie les tendances observées lors de l'exploration des données.

**3. L'importance des variables d'une Random Forest implique-t-elle nécessairement une causalité ?**

Non. Une forte importance indique seulement qu'une variable contribue fortement aux prédictions du modèle. Cela ne signifie pas que cette variable est la cause directe de la souscription. Une variable peut être très prédictive parce qu'elle est corrélée à d'autres facteurs explicatifs ou parce qu'elle capture une information utile pour la classification. L'importance mesurée par la Random Forest doit donc être interprétée comme une mesure de contribution prédictive et non comme une preuve de causalité.

## 8. Comparaison statistique par validation croisée répétée

On compare les modèles à l'aide d'une validation croisée stratifiée répétée. Pour limiter le temps de calcul, on utilise ici 5 plis répétés 2 fois.

Pour un rapport plus robuste, on peut utiliser 10 plis répétés 5 fois.


In [ ]:
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_results = {}
for name, pipe in pipelines.items():
    print("Validation croisée :", name)
    scores = cross_validate(pipe, X, y, cv=cv, scoring=scoring, n_jobs=-1, return_train_score=False)
    cv_results[name] = scores

summary_rows = []
for name, scores in cv_results.items():
    row = {"Modèle": name}
    for metric in scoring.keys():
        values = scores[f"test_{metric}"]
        row[f"{metric}_moyenne"] = values.mean()
        row[f"{metric}_ecart_type"] = values.std()
    summary_rows.append(row)

cv_summary = pd.DataFrame(summary_rows).sort_values("f1_moyenne", ascending=False)
cv_summary


In [ ]:
# Visualisation des F1-scores en validation croisée
plt.figure(figsize=(10, 5))
plt.boxplot([cv_results[name]["test_f1"] for name in pipelines.keys()], labels=list(pipelines.keys()), vert=False)
plt.title("Distribution des F1-scores en validation croisée")
plt.xlabel("F1-score")
plt.show()


### Test de Friedman

Le test de Friedman compare globalement plusieurs algorithmes évalués sur les mêmes plis de validation croisée.

Hypothèses :

$$
H_0 : \text{les modèles ont des performances équivalentes.}
$$

$$
H_1 : \text{au moins un modèle a une performance différente.}
$$


In [ ]:
# Test de Friedman sur les F1-scores
f1_arrays = [cv_results[name]["test_f1"] for name in pipelines.keys()]
stat, p_value = friedmanchisquare(*f1_arrays)

print("Test de Friedman sur le F1-score")
print("Statistique =", stat)
print("p-value =", p_value)

if p_value < 0.05:
    print("Conclusion : on rejette H0. Il existe une différence significative entre au moins deux modèles.")
else:
    print("Conclusion : on ne rejette pas H0. Les différences observées ne sont pas significatives au seuil 5%.")


### Comparaisons deux à deux avec Wilcoxon

On compare le meilleur modèle selon le F1-score moyen aux autres modèles. On applique une correction de Bonferroni simple.


In [ ]:
best_model = cv_summary.iloc[0]["Modèle"]
print("Meilleur modèle selon F1 moyen :", best_model)

alpha = 0.05
m = len(pipelines) - 1
alpha_bonf = alpha / m

wilcoxon_rows = []
for name in pipelines.keys():
    if name == best_model:
        continue
    stat, p = wilcoxon(cv_results[best_model]["test_f1"], cv_results[name]["test_f1"])
    wilcoxon_rows.append({
        "Comparaison": f"{best_model} vs {name}",
        "p_value": p,
        "significatif_Bonferroni": p < alpha_bonf
    })

pd.DataFrame(wilcoxon_rows).sort_values("p_value")


### Questions

1. Le test de Friedman rejette-t-il l'hypothèse d'égalité des performances ?
2. Le meilleur modèle est-il significativement meilleur que tous les autres ?
3. Pourquoi faut-il corriger le seuil lorsqu'on fait plusieurs tests deux à deux ?
4. Pourquoi une différence statistiquement significative n'est-elle pas toujours une différence importante en pratique ?


### Réponses aux questions

**1. Le test de Friedman rejette-t-il l'hypothèse d'égalité des performances ?**

Oui. Le test de Friedman met en évidence des différences significatives entre les performances des modèles évalués sur les mêmes plis de validation croisée. L'hypothèse nulle d'égalité des performances est donc rejetée, ce qui signifie qu'au moins un modèle se comporte différemment des autres.

**2. Le meilleur modèle est-il significativement meilleur que tous les autres ?**

Non. Les comparaisons deux à deux réalisées avec le test de Wilcoxon montrent que la Ridge Logistic est significativement meilleure que la plupart des autres modèles. En revanche, la différence observée avec le SVM RBF balanced n'est pas statistiquement significative après correction de Bonferroni. On ne peut donc pas conclure que la Ridge Logistic soit supérieure à ce modèle avec un niveau de confiance suffisant.

**3. Pourquoi faut-il corriger le seuil lorsqu'on fait plusieurs tests deux à deux ?**

Lorsque plusieurs comparaisons sont réalisées simultanément, le risque de trouver une différence significative par simple hasard augmente. La correction de Bonferroni permet de contrôler ce risque en utilisant un seuil plus strict pour chaque test individuel. Cela limite la probabilité de conclure à tort qu'une différence existe alors qu'elle est due au hasard.

**4. Pourquoi une différence statistiquement significative n'est-elle pas toujours une différence importante en pratique ?**

La significativité statistique indique qu'une différence est peu susceptible d'être due au hasard, mais elle ne renseigne pas sur son ampleur. Deux modèles peuvent présenter une différence statistiquement significative tout en ayant des performances très proches. Dans un contexte opérationnel, il faut également tenir compte de critères pratiques tels que la simplicité du modèle, le temps de calcul, l'interprétabilité ou encore le coût des erreurs de classification.

### Test de McNemar

En complément des mesures de performance classiques, on utilisera le test de
McNemar pour comparer statistiquement les classifieurs deux à deux sur le jeu
de test. Ce test est adapté car les modèles sont évalués sur les mêmes
observations. Il permet de déterminer si les différences observées entre deux
classifieurs sont statistiquement significatives ou si elles peuvent être
attribuées aux fluctuations de l'échantillon de test.

## Comparaison statistique de deux classifieurs : test de McNemar

Lorsque deux classifieurs sont évalués sur le même jeu de test, leurs erreurs
ne sont pas indépendantes. On ne doit donc pas comparer uniquement leurs
accuracies comme si elles provenaient de deux échantillons indépendants.

Le test de McNemar permet de comparer deux classifieurs sur les mêmes
observations. Il s'intéresse uniquement aux observations pour lesquelles les
deux modèles ne se comportent pas de la même manière.

Pour deux modèles $A$ et $B$, on construit le tableau suivant :

$$
\begin{array}{c|cc}
 & B \text{ correct} & B \text{ incorrect} \\
\hline
A \text{ correct} & n_{00} & n_{01} \\
A \text{ incorrect} & n_{10} & n_{11}
\end{array}
$$

Les quantités importantes sont :

$$
n_{01}
$$

le nombre d'observations correctement classées par $A$ mais mal classées
par \(B\), et

$$
n_{10}
$$

le nombre d'observations mal classées par $A$ mais correctement classées
par $B$.

L'hypothèse nulle du test est :

$$
H_0 : n_{01} = n_{10}.
$$

Autrement dit, les deux classifieurs ont une probabilité d'erreur comparable
sur les observations où ils diffèrent.

L'hypothèse alternative est :

$$
H_1 : n_{01} \neq n_{10}.
$$

Si la p-valeur est petite, on rejette \(H_0\), ce qui indique que les deux
classifieurs ont des comportements significativement différents sur le jeu de
test.

### Code Python pour comparer deux modèles
Ce code suppose que tu as déjà entraîné plusieurs modèles et stocké leurs prédictions dans un dictionnaire du type :

In [ ]:
test_predictions = {
    r["Modèle"]: r["y_pred"]
    for r in results
}

test_scores = {
    r["Modèle"]: r["y_score"]
    for r in results
}

trained_pipelines = {
    r["Modèle"]: r["pipe"]
    for r in results
}

print(test_predictions.keys())

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.stats.contingency_tables import mcnemar

def mcnemar_table(y_true, y_pred_a, y_pred_b):
    """
    Construit la table de contingence pour le test de McNemar.
    """
    a_correct = (y_pred_a == y_true)
    b_correct = (y_pred_b == y_true)

    n00 = np.sum(a_correct & b_correct)
    n01 = np.sum(a_correct & ~b_correct)
    n10 = np.sum(~a_correct & b_correct)
    n11 = np.sum(~a_correct & ~b_correct)

    table = np.array([
        [n00, n01],
        [n10, n11]
    ])

    return table


def compare_two_classifiers_mcnemar(y_true, y_pred_a, y_pred_b, name_a, name_b):
    """
    Compare deux classifieurs avec le test de McNemar.
    """
    table = mcnemar_table(y_true, y_pred_a, y_pred_b)

    result = mcnemar(
        table,
        exact=True
    )

    print(f"Comparaison : {name_a} vs {name_b}")
    print("Table de McNemar :")
    print(pd.DataFrame(
        table,
        index=[f"{name_a} correct", f"{name_a} incorrect"],
        columns=[f"{name_b} correct", f"{name_b} incorrect"]
    ))

    print("\nStatistique :", result.statistic)
    print("p-valeur :", result.pvalue)

    if result.pvalue < 0.05:
        print("\nConclusion : différence significative entre les deux classifieurs.")
    else:
        print("\nConclusion : pas de différence significative détectée.")

    return table, result

    

Exemple :

In [ ]:
print(test_predictions.keys())

In [ ]:
test_predictions["SVM"] = test_predictions["SVM RBF balanced"]

table_rf_svm, result_rf_svm = compare_two_classifiers_mcnemar(
    y_test,
    test_predictions["Random Forest"],
    test_predictions["SVM"],
    "Random Forest",
    "SVM"
)

Comparer tous les classifieurs deux à deux

In [ ]:
from itertools import combinations

mcnemar_results = []

for model_a, model_b in combinations(test_predictions.keys(), 2):
    y_pred_a = test_predictions[model_a]
    y_pred_b = test_predictions[model_b]

    table = mcnemar_table(y_test, y_pred_a, y_pred_b)

    result = mcnemar(
        table,
        exact=True
    )

    n01 = table[0, 1]
    n10 = table[1, 0]

    mcnemar_results.append({
        "Modèle A": model_a,
        "Modèle B": model_b,
        "A correct / B incorrect": n01,
        "A incorrect / B correct": n10,
        "statistique": result.statistic,
        "p_value": result.pvalue
    })

mcnemar_results_df = pd.DataFrame(mcnemar_results)
mcnemar_results_df.sort_values("p_value")

Correction pour comparaisons multiples

### Rappel : pourquoi corriger les p-valeurs ?

Lorsque l'on compare deux classifieurs avec le test de McNemar, on effectue
un test statistique avec une hypothèse nulle :

$$
H_0 : \text{les deux classifieurs ont des performances équivalentes.}
$$

Si la p-valeur est inférieure à un seuil, par exemple

$$
\alpha = 0.05,
$$

on rejette $H_0$ et on conclut qu'il existe une différence significative
entre les deux classifieurs.

Cependant, dans notre étude, on ne compare pas seulement deux modèles. On
compare plusieurs modèles deux à deux : Naive Bayes, CART, Random Forest,
SVM, réseau de neurones, régression logistique Ridge, etc.

Si l'on réalise beaucoup de tests avec un seuil de $5\%$, le risque de
trouver au moins une différence significative par hasard augmente. Par
exemple, même si tous les modèles avaient en réalité des performances
équivalentes, certaines comparaisons pourraient donner une p-valeur inférieure
à $0.05$ uniquement à cause des fluctuations de l'échantillon de test.

On parle alors de problème des comparaisons multiples.

Pour limiter ce risque, on corrige les p-valeurs. Une méthode classique est
la correction de Holm. Elle ajuste les p-valeurs afin de contrôler le risque
global de conclure à tort qu'il existe une différence significative.

Ainsi, après correction, une comparaison est considérée comme significative
seulement si sa p-valeur corrigée reste inférieure au seuil choisi :

$$
p_{\text{corrigée}} < 0.05.
$$

Dans la suite, on applique donc une correction de Holm aux p-valeurs issues
des tests de McNemar réalisés entre les différents classifieurs.

Comme on compare plusieurs modèles deux à deux, on doit corriger les p-valeurs. Par exemple avec la correction de Holm.

In [ ]:
from statsmodels.stats.multitest import multipletests

p_values = mcnemar_results_df["p_value"].values

reject, pvals_corrected, _, _ = multipletests(
    p_values,
    alpha=0.05,
    method="holm"
)

mcnemar_results_df["p_value_corrigee_Holm"] = pvals_corrected
mcnemar_results_df["difference_significative"] = reject

mcnemar_results_df.sort_values("p_value_corrigee_Holm")

### Questions

1. Pourquoi le test de McNemar est-il plus adapté qu'une simple comparaison
   des accuracies lorsque deux modèles sont évalués sur le même jeu de test ?

2. Que représentent les quantités $n_{01}$ et $n_{10}$ dans la table de
   McNemar ?

3. Quelle est l'hypothèse nulle du test de McNemar ?

4. Si un modèle $A$ a une accuracy légèrement supérieure à un modèle $B$,
   cela signifie-t-il forcément que la différence est statistiquement
   significative ? Justifier.

5. Pourquoi faut-il corriger les p-valeurs lorsqu'on compare plusieurs modèles
   deux à deux ?

6. Après correction de Holm, quels modèles sont significativement différents ?

7. Le modèle ayant la meilleure accuracy est-il aussi significativement
   meilleur que tous les autres modèles ?

### Réponses aux questions

**1. Pourquoi le test de McNemar est-il plus adapté qu'une simple comparaison des accuracies lorsque deux modèles sont évalués sur le même jeu de test ?**

Une simple comparaison des accuracies ne tient compte que du nombre total de prédictions correctes. Le test de McNemar utilise au contraire les prédictions individuelles de chaque modèle et s'intéresse aux observations sur lesquelles les deux modèles sont en désaccord. Il permet ainsi de déterminer si la différence observée entre les performances est réellement significative ou si elle peut être attribuée au hasard.

**2. Que représentent les quantités \(n_{01}\) et \(n_{10}\) dans la table de McNemar ?**

La quantité \(n_{01}\) représente le nombre d'observations correctement classées par le modèle A mais incorrectement classées par le modèle B. À l'inverse, \(n_{10}\) représente le nombre d'observations incorrectement classées par le modèle A mais correctement classées par le modèle B. Ce sont ces deux quantités qui sont utilisées par le test de McNemar.

**3. Quelle est l'hypothèse nulle du test de McNemar ?**

L'hypothèse nulle (\(H_0\)) est que les deux modèles ont la même probabilité de commettre une erreur. Autrement dit, les désaccords entre les deux modèles sont symétriques et aucune différence significative de performance n'existe entre eux.

**4. Si un modèle \(A\) a une accuracy légèrement supérieure à un modèle \(B\), cela signifie-t-il forcément que la différence est statistiquement significative ? Justifier.**

Non. Une différence d'accuracy peut être due au hasard, en particulier lorsque les modèles commettent des erreurs très similaires. Le test de McNemar permet précisément de vérifier si la différence observée est suffisamment importante pour être considérée comme statistiquement significative.

**5. Pourquoi faut-il corriger les p-valeurs lorsqu'on compare plusieurs modèles deux à deux ?**

Lorsque de nombreuses comparaisons sont réalisées simultanément, la probabilité d'obtenir au moins un résultat significatif par hasard augmente. Les corrections de Bonferroni ou de Holm permettent de contrôler ce risque et d'éviter les faux positifs.

**6. Après correction de Holm, quels modèles sont significativement différents ?**

Après correction de Holm, plusieurs différences restent significatives. En particulier, Naive Bayes est significativement différent de tous les autres modèles. Les modèles équilibrés (Ridge Logistic, SVM linéaire balanced et SVM RBF balanced) sont également significativement différents des SVM non pondérés et du Random Forest dans plusieurs comparaisons. En revanche, certaines comparaisons entre les meilleurs modèles ne sont plus significatives après correction.

**7. Le modèle ayant la meilleure accuracy est-il aussi significativement meilleur que tous les autres modèles ?**

Non. Le modèle ayant la meilleure accuracy n'est pas significativement meilleur que tous les autres modèles après correction de Holm. De plus, les modèles présentant la meilleure accuracy ne sont pas forcément ceux qui obtiennent les meilleurs rappels ou F1-scores. Dans ce problème déséquilibré, les modèles pondérés comme la Ridge Logistic ou les SVM balanced sont souvent préférables malgré une accuracy légèrement plus faible.

### Variante si comparaison un modèle de référence aux autres

Par exemple, si Random Forest est le meilleur modèle selon le F1-score ou l’AUC, on peut le comparer aux autres :

In [ ]:
reference_model = "Random Forest"

mcnemar_vs_reference = []

for model_name in test_predictions.keys():
    if model_name == reference_model:
        continue

    table = mcnemar_table(
        y_test,
        test_predictions[reference_model],
        test_predictions[model_name]
    )

    result = mcnemar(table, exact=True)

    mcnemar_vs_reference.append({
        "Modèle de référence": reference_model,
        "Modèle comparé": model_name,
        "Référence correct / autre incorrect": table[0, 1],
        "Référence incorrect / autre correct": table[1, 0],
        "p_value": result.pvalue
    })

mcnemar_vs_reference_df = pd.DataFrame(mcnemar_vs_reference)

reject, pvals_corrected, _, _ = multipletests(
    mcnemar_vs_reference_df["p_value"].values,
    alpha=0.05,
    method="holm"
)

mcnemar_vs_reference_df["p_value_corrigee_Holm"] = pvals_corrected
mcnemar_vs_reference_df["difference_significative"] = reject

mcnemar_vs_reference_df.sort_values("p_value_corrigee_Holm")

## 9. Partie théorique sur les modèles

### Naive Bayes

Le classifieur Naive Bayes repose sur le théorème de Bayes :

$$
P(Y=c \mid X=x)=\frac{P(X=x \mid Y=c)P(Y=c)}{P(X=x)}.
$$

L'hypothèse naïve consiste à supposer que les variables explicatives sont indépendantes conditionnellement à la classe.

### CART

Un arbre CART partitionne récursivement l'espace des variables en choisissant les séparations qui réduisent le plus l'impureté, par exemple l'indice de Gini :

$$
Gini(t)=1-\sum_{k=1}^K p_{k,t}^2.
$$

### Random Forest

Une Random Forest combine plusieurs arbres entraînés sur des échantillons bootstrap. À chaque séparation, un sous-ensemble aléatoire de variables est testé, ce qui réduit la corrélation entre arbres.

### SVM

Le SVM cherche un hyperplan qui maximise la marge entre les classes. Dans le cas à marge souple, le problème primal s'écrit :

$$
\min_{w,b,\xi} \frac{1}{2}\lVert w\rVert^2 + C\sum_{i=1}^n \xi_i
$$

sous les contraintes :

$$
y_i(w^Tx_i+b)\geq 1-\xi_i, \qquad \xi_i\geq 0.
$$

### Réseau de neurones

Un réseau de neurones multicouche apprend une fonction non linéaire par composition de transformations affines et de fonctions d'activation.

### Régression logistique Ridge

La régression logistique Ridge minimise une perte logistique pénalisée par la norme \(L_2\) des coefficients :

$$
\min_\beta -\sum_{i=1}^n \left[y_i\log(p_i)+(1-y_i)\log(1-p_i)\right]
+\lambda\sum_{j=1}^p \beta_j^2.
$$



## 10. Biais-variance, risque réel, risque empirique et généralisation

Cette partie relie les expériences précédentes aux notions d'apprentissage statistique.

### Risque réel et risque empirique

Soit une variable aléatoire \((X,Y)\) distribuée selon une loi inconnue $P$. Pour un classifieur $f$, le **risque réel** est

$$
R(f)=\mathbb{E}_{(X,Y)\sim P}\left[\ell(f(X),Y)\right],
$$

où $\ell$ est une fonction de perte. En classification binaire, on utilise souvent la perte 0-1 :

$$
\ell(f(X),Y)=\mathbb{1}_{f(X)\neq Y}.
$$

Comme la loi $P$ est inconnue, on minimise en pratique le **risque empirique** calculé sur l'échantillon d'apprentissage :

$$
\widehat{R}_n(f)=\frac{1}{n}\sum_{i=1}^n \ell(f(x_i),y_i).
$$

L'objectif de l'apprentissage statistique n'est donc pas seulement d'obtenir une faible erreur d'apprentissage, mais d'obtenir un faible risque réel. L'écart

$$
R(f)-\widehat{R}_n(f)
$$

est appelé **écart de généralisation**.

### Compromis biais-variance

Un modèle trop simple a souvent un **biais élevé** : il ne capture pas suffisamment la structure des données. Il sous-apprend. Par exemple, un arbre très peu profond ou un SVM linéaire peuvent être trop simples.

Un modèle trop complexe a souvent une **variance élevée** : il s'adapte trop aux fluctuations de l'échantillon d'apprentissage. Il sur-apprend. Par exemple, un arbre très profond, un SVM RBF avec un grand \(C\) et un grand \(\gamma\), ou un réseau de neurones trop flexible peuvent avoir une variance élevée.

Dans le cas d'une perte quadratique, on écrit classiquement :

$$
\mathbb{E}\left[(Y-\widehat{f}(X))^2\right]=
\text{biais}^2+\text{variance}+\text{bruit irréductible}.
$$

En classification, cette décomposition n'est pas toujours aussi directe, mais l'idée reste la même : on cherche un modèle assez flexible pour apprendre la structure, mais pas trop flexible pour éviter le sur-apprentissage.

### Lien avec les SVM

Dans un SVM, le paramètre $C$ contrôle le compromis entre marge large et erreurs d'apprentissage :

$$
\min_{w,b,\xi}
\frac{1}{2}\|w\|^2+C\sum_{i=1}^n \xi_i.
$$

Un petit $C$ accepte davantage d'erreurs mais favorise une grande marge : le modèle est plus régularisé, donc généralement moins variable.

Un grand $C$ pénalise fortement les erreurs : le modèle cherche à classer correctement les points d'apprentissage, ce qui peut augmenter la variance.

Avec un noyau RBF, le paramètre $\gamma$ joue aussi un rôle important. Un grand $\gamma$ rend la frontière très locale et flexible, ce qui peut conduire au sur-apprentissage.

### Borne leave-one-out et vecteurs supports

Pour un SVM à marge dure dans un cas séparable, si l'on retire un point qui n'est pas vecteur support, l'hyperplan optimal ne change pas. Ce point sera donc encore correctement classé. Les erreurs leave-one-out ne peuvent alors venir que des vecteurs supports. On obtient l'idée suivante :

$$
R_{LOO} \leq \frac{\#\text{vecteurs supports}}{n}.
$$

Cette borne donne une intuition importante : si un SVM utilise beaucoup de vecteurs supports, il peut être plus sensible à l'échantillon d'apprentissage et donc avoir une variance plus élevée.

Dans le cas à marge souple et sur des données non parfaitement séparables, cette borne doit être interprétée avec prudence, mais le nombre de vecteurs supports reste un indicateur utile de la complexité effective du modèle.



### Questions théoriques

1. Quelle est la différence entre risque réel et risque empirique ?
2. Pourquoi ne peut-on pas calculer exactement le risque réel ?
3. Que signifie un modèle à fort biais ? Donner un exemple parmi les modèles étudiés.
4. Que signifie un modèle à forte variance ? Donner un exemple parmi les modèles étudiés.
5. Dans un SVM, quel est l'effet d'un grand $C$ ?
6. Dans un SVM RBF, quel est l'effet d'un grand $\gamma$ ?
7. Pourquoi le nombre de vecteurs supports donne-t-il une indication sur la complexité du modèle ?
8. Pourquoi la validation leave-one-out est-elle liée à la capacité de généralisation ?


### Réponses aux questions théoriques

**1. Quelle est la différence entre risque réel et risque empirique ?**

Le risque empirique correspond à l'erreur mesurée sur les données d'apprentissage utilisées pour entraîner le modèle. Le risque réel correspond quant à lui à l'erreur que le modèle commettrait sur l'ensemble de la population ou sur de nouvelles données provenant de la même distribution. L'objectif de l'apprentissage statistique est de minimiser le risque réel, même si seul le risque empirique est directement observable.

**2. Pourquoi ne peut-on pas calculer exactement le risque réel ?**

Le risque réel dépend de la distribution de probabilité inconnue qui génère les données. Comme cette distribution n'est jamais observée dans sa totalité, il est impossible de calculer exactement le risque réel. On utilise donc des estimations obtenues à partir de jeux de test ou de procédures de validation croisée.

**3. Que signifie un modèle à fort biais ? Donner un exemple parmi les modèles étudiés.**

Un modèle à fort biais est un modèle trop simple pour capturer correctement les relations présentes dans les données. Il a tendance à sous-apprendre et à obtenir des performances faibles aussi bien sur les données d'entraînement que sur les nouvelles données. Parmi les modèles étudiés, Naive Bayes constitue un bon exemple de modèle à fort biais, car ses hypothèses simplificatrices limitent sa capacité à représenter des relations complexes.

**4. Que signifie un modèle à forte variance ? Donner un exemple parmi les modèles étudiés.**

Un modèle à forte variance est très sensible aux données d'entraînement. Il peut obtenir d'excellentes performances sur l'échantillon d'apprentissage mais généraliser moins bien sur de nouvelles données. Dans nos expériences, certains SVM RBF avec des valeurs élevées de \(C\) et de \(\gamma\) illustrent ce phénomène : ils obtiennent des scores d'entraînement très élevés mais des performances de validation plus faibles, ce qui traduit un sur-apprentissage.

**5. Dans un SVM, quel est l'effet d'un grand \(C\) ?**

Un grand \(C\) pénalise fortement les erreurs de classification sur les données d'entraînement. Le modèle cherche alors à classer correctement un maximum d'observations, quitte à construire une frontière de décision plus complexe. Cela réduit généralement le biais mais peut augmenter la variance et le risque de sur-apprentissage.

**6. Dans un SVM RBF, quel est l'effet d'un grand \(\gamma\) ?**

Un grand \(\gamma\) rend l'influence de chaque observation plus locale. La frontière de décision devient alors plus flexible et plus irrégulière. Le modèle peut mieux s'adapter aux données d'entraînement, mais il devient également plus sensible au bruit et donc plus susceptible de sur-apprendre.

**7. Pourquoi le nombre de vecteurs supports donne-t-il une indication sur la complexité du modèle ?**

Les vecteurs supports sont les observations qui déterminent la position de la frontière de décision. Lorsqu'un grand nombre d'observations deviennent des vecteurs supports, cela signifie que la frontière dépend fortement des données d'entraînement. Le modèle est alors généralement plus complexe et potentiellement plus sensible aux variations de l'échantillon.

**8. Pourquoi la validation leave-one-out est-elle liée à la capacité de généralisation ?**

La validation leave-one-out consiste à entraîner le modèle sur toutes les observations sauf une, puis à tester sa capacité à prédire l'observation laissée de côté. Cette procédure évalue directement la capacité du modèle à généraliser à des données qu'il n'a jamais vues. Une faible erreur leave-one-out indique donc généralement une bonne capacité de généralisation.


### Étude pratique : courbes d'apprentissage

Les courbes d'apprentissage permettent de comparer le score d'apprentissage et le score de validation lorsque la taille de l'échantillon d'apprentissage augmente.

- Si les deux scores sont faibles et proches, le modèle souffre probablement de **biais élevé**.
- Si le score d'apprentissage est élevé mais le score de validation est nettement plus faible, le modèle souffre probablement de **variance élevée**.
- Si l'écart diminue lorsque la taille d'apprentissage augmente, davantage de données peuvent améliorer la généralisation.


In [ ]:

from sklearn.model_selection import StratifiedKFold, learning_curve, validation_curve, LeaveOneOut, cross_val_score

cv_lc = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
train_sizes = np.linspace(0.1, 1.0, 5)

models_to_analyze = [
    "CART",
    "Random Forest",
    "SVM RBF",
    "Ridge Logistic"
]

plt.figure(figsize=(10, 7))

for name in models_to_analyze:
    train_sizes_abs, train_scores, val_scores = learning_curve(
        pipelines[name],
        X_train,
        y_train,
        train_sizes=train_sizes,
        cv=cv_lc,
        scoring="f1",
        n_jobs=-1
    )
    
    train_mean = train_scores.mean(axis=1)
    val_mean = val_scores.mean(axis=1)
    
    plt.plot(train_sizes_abs, train_mean, marker="o", linestyle="--", label=f"{name} - train")
    plt.plot(train_sizes_abs, val_mean, marker="o", label=f"{name} - validation")

plt.xlabel("Taille de l'ensemble d'apprentissage")
plt.ylabel("F1-score")
plt.title("Courbes d'apprentissage : biais, variance et généralisation")
plt.legend()
plt.grid(alpha=0.3)
plt.show()



### Questions pratiques sur les courbes d'apprentissage

1. Quel modèle présente le plus grand écart entre score d'apprentissage et score de validation ?
2. Quel modèle semble le plus stable ?
3. Observe-t-on un cas de sous-apprentissage ?
4. Observe-t-on un cas de sur-apprentissage ?
5. Les modèles bénéficient-ils d'une augmentation de la taille de l'échantillon ?


### Réponses aux questions pratiques sur les courbes d'apprentissage

**1. Quel modèle présente le plus grand écart entre score d'apprentissage et score de validation ?**

Le **Random Forest** présente l'écart le plus important. Son F1-score d'entraînement reste proche de 1 quelle que soit la taille de l'échantillon, alors que son F1-score de validation reste nettement plus faible (entre 0.30 et 0.40). Cet écart important indique que le modèle s'adapte très fortement aux données d'entraînement.

**2. Quel modèle semble le plus stable ?**

La **Ridge Logistic** semble être le modèle le plus stable. Les courbes d'entraînement et de validation sont relativement proches et convergent lorsque la taille de l'échantillon augmente. De plus, les performances de validation restent élevées et peu sensibles aux variations de l'échantillon.

**3. Observe-t-on un cas de sous-apprentissage ?**

Non

**4. Observe-t-on un cas de sur-apprentissage ?**

Oui. Le **Random Forest** 

**5. Les modèles bénéficient-ils d'une augmentation de la taille de l'échantillon ?**

Oui, dans l'ensemble les performances de validation augmentent lorsque la taille de l'échantillon d'apprentissage devient plus importante. Cette amélioration est particulièrement visible pour le **CART** et le **SVM RBF**, dont les scores de validation progressent sensiblement. Pour la **Ridge Logistic**, le gain est plus modéré mais la stabilité du modèle s'améliore. En revanche, l'augmentation de la taille de l'échantillon ne suffit pas à éliminer complètement le sur-apprentissage observé pour le Random Forest.


### Étude pratique : influence de \(C\) sur un SVM RBF

On fait varier le paramètre \(C\) du SVM RBF. Un petit \(C\) donne un modèle plus régularisé. Un grand \(C\) donne un modèle plus flexible, qui cherche davantage à corriger les erreurs d'apprentissage.


In [ ]:

C_values = np.logspace(-2, 2, 5)

svm_rbf_for_curve = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", SVC(kernel="rbf", gamma="scale", class_weight="balanced", random_state=42))
])

train_scores, val_scores = validation_curve(
    svm_rbf_for_curve,
    X_train,
    y_train,
    param_name="model__C",
    param_range=C_values,
    cv=cv_lc,
    scoring="f1",
    n_jobs=-1
)

plt.figure(figsize=(8, 5))
plt.semilogx(C_values, train_scores.mean(axis=1), marker="o", linestyle="--", label="train")
plt.semilogx(C_values, val_scores.mean(axis=1), marker="o", label="validation")
plt.xlabel("Paramètre C")
plt.ylabel("F1-score")
plt.title("Validation curve du SVM RBF : effet de C")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

pd.DataFrame({
    "C": C_values,
    "F1 train moyen": train_scores.mean(axis=1),
    "F1 validation moyen": val_scores.mean(axis=1),
    "écart train-validation": train_scores.mean(axis=1) - val_scores.mean(axis=1)
})



### Interprétation attendue

Lorsque \(C\) augmente, le score d'apprentissage peut augmenter, car le modèle accepte moins d'erreurs sur les données d'entraînement. Si le score de validation n'augmente pas ou diminue, cela indique que l'on gagne surtout en adaptation à l'échantillon, pas en généralisation.

Le meilleur \(C\) n'est donc pas nécessairement celui qui donne le meilleur score d'apprentissage, mais celui qui donne le meilleur score de validation.



### Étude pratique : vecteurs supports et leave-one-out

On estime maintenant, sur un sous-échantillon pour garder un temps de calcul raisonnable, le nombre de vecteurs supports d'un SVM linéaire et on le compare à l'erreur leave-one-out.

L'idée théorique est la suivante : dans un SVM séparable à marge dure, les points qui ne sont pas vecteurs supports ne changent pas l'hyperplan lorsqu'ils sont retirés. Les erreurs leave-one-out sont donc contrôlées par le nombre de vecteurs supports.


In [ ]:

# Sous-échantillon stratifié pour rendre le leave-one-out raisonnable en temps de calcul
n_yes = min(40, int((y_train == 1).sum()))
n_no = min(160, int((y_train == 0).sum()))

idx_yes = y_train[y_train == 1].sample(n=n_yes, random_state=42).index
idx_no = y_train[y_train == 0].sample(n=n_no, random_state=42).index
idx_loo = idx_yes.union(idx_no)

X_loo = X_train.loc[idx_loo]
y_loo = y_train.loc[idx_loo]

svm_linear_loo = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", SVC(kernel="linear", C=1.0, class_weight="balanced", random_state=42))
])

svm_linear_loo.fit(X_loo, y_loo)

n_support_vectors = svm_linear_loo.named_steps["model"].n_support_.sum()
n_total = len(y_loo)
sv_ratio = n_support_vectors / n_total

loo = LeaveOneOut()
loo_scores = cross_val_score(
    svm_linear_loo,
    X_loo,
    y_loo,
    cv=loo,
    scoring="accuracy",
    n_jobs=-1
)

loo_error = 1 - loo_scores.mean()

print("Nombre d'observations utilisées :", n_total)
print("Nombre de vecteurs supports :", n_support_vectors)
print("Proportion de vecteurs supports :", round(sv_ratio, 4))
print("Erreur leave-one-out empirique :", round(loo_error, 4))



### Questions sur le leave-one-out et les SVM

1. Quelle est la proportion de vecteurs supports obtenue ?
2. L'erreur leave-one-out est-elle inférieure ou proche de cette proportion ?
3. Que peut-on conclure si presque tous les points deviennent vecteurs supports ?
4. Pourquoi cette borne est-elle plus claire dans le cas d'un SVM à marge dure séparable que dans le cas réel de données bruitées ?
5. En quoi cette expérience illustre-t-elle le lien entre complexité du modèle, variance et généralisation ?


### Réponses

**1. Quelle est la proportion de vecteurs supports obtenue ?**

Le modèle SVM linéaire utilise 65 vecteurs supports sur 200 observations, soit une proportion de :

\[
\frac{65}{200}=0.325
\]

La proportion de vecteurs supports est donc d'environ **32,5 %**.

---

**2. L'erreur leave-one-out est-elle inférieure ou proche de cette proportion ?**

L'erreur leave-one-out obtenue est de **20,5 %**, tandis que la proportion de vecteurs supports est de **32,5 %**.

L'erreur leave-one-out est donc **inférieure** à la proportion de vecteurs supports, ce qui est cohérent avec l'idée théorique selon laquelle le nombre de vecteurs supports fournit une borne supérieure approximative du risque leave-one-out.

---

**3. Que peut-on conclure si presque tous les points deviennent vecteurs supports ?**

Si presque toutes les observations deviennent des vecteurs supports, cela signifie que le modèle dépend fortement des données d'entraînement. La frontière de décision devient plus sensible aux variations de l'échantillon, ce qui traduit généralement une **variance élevée** et un risque accru de **sur-apprentissage**.

---

**4. Pourquoi cette borne est-elle plus claire dans le cas d'un SVM à marge dure séparable que dans le cas réel de données bruitées ?**

Dans un SVM à marge dure et parfaitement séparable, seuls les vecteurs supports déterminent l'hyperplan optimal. Les points non supports n'ont aucun effet sur la frontière de décision lorsqu'ils sont retirés.

Dans les données réelles, les observations contiennent souvent du bruit et ne sont pas parfaitement séparables. On utilise alors une marge souple, ce qui rend la relation entre vecteurs supports et erreur de généralisation moins directe. La borne doit donc être interprétée comme une indication plutôt que comme un résultat exact.

---

**5. En quoi cette expérience illustre-t-elle le lien entre complexité du modèle, variance et généralisation ?**

Le nombre de vecteurs supports peut être vu comme un indicateur de la complexité effective du SVM. Plus cette proportion est élevée, plus le modèle est susceptible d'être sensible aux fluctuations de l'échantillon d'apprentissage, ce qui augmente la variance.

À l'inverse, lorsque le nombre de vecteurs supports reste modéré et que l'erreur leave-one-out demeure faible, le modèle présente une meilleure capacité de généralisation. Cette expérience illustre donc le compromis classique entre **complexité**, **variance** et **performance sur de nouvelles données**.

## 11. Option : règles d'association

Cette partie utilise les variables catégorielles pour chercher des règles du type :

$$
\{job=admin., contact=cellular\}\Rightarrow \{y=yes\}.
$$

Les mesures principales sont :

$$
\operatorname{support}(A\Rightarrow B)=P(A\cap B),
$$

$$
\operatorname{confidence}(A\Rightarrow B)=P(B\mid A),
$$

$$
\operatorname{lift}(A\Rightarrow B)=frac{P(A\cap B)}{P(A)P(B)}.
$$

Cette partie est optionnelle. Elle nécessite le package `mlxtend`. Si le package n'est pas disponible, le reste du notebook peut être exécuté normalement.



In [ ]:
#%pip install mlxtend

In [ ]:
# Partie optionnelle : règles d'association avec Apriori
# Cette cellule est robuste : elle vérifie que les données existent et ne relance pas pip automatiquement.
# Si mlxtend n'est pas installé, exécuter dans une cellule séparée : %pip install mlxtend

# Règles d'association avec Apriori
# Cette cellule est rendue autonome : si df n'existe pas encore,
# on recharge les données Bank Marketing.

import pandas as pd

# ---------------------------------------------------------------------
# 1. Vérifier si df existe, sinon charger les données
# ---------------------------------------------------------------------

if "df" not in globals():
    print("Le DataFrame df n'existe pas encore. Chargement des données Bank Marketing...")

    import zipfile
    import io
    import requests

    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank-additional.zip"

    response = requests.get(url)
    z = zipfile.ZipFile(io.BytesIO(response.content))

    with z.open("bank-additional/bank-additional.csv") as f:
        df = pd.read_csv(f, sep=";")

    print("Données chargées :", df.shape)

else:
    print("DataFrame df déjà disponible :", df.shape)


# ---------------------------------------------------------------------
# 2. Importer mlxtend
# ---------------------------------------------------------------------

from mlxtend.frequent_patterns import apriori, association_rules
MLXTEND_AVAILABLE = True


# ---------------------------------------------------------------------
# 3. Extraction des règles d'association
# ---------------------------------------------------------------------

if MLXTEND_AVAILABLE:

    if "y" not in df.columns:
        raise ValueError("La colonne cible 'y' est absente du DataFrame df.")

    # Variables catégorielles
    categorical_features_assoc = df.select_dtypes(
        include=["object", "category", "bool"]
    ).columns.tolist()

    # On retire la cible des variables explicatives si elle est présente
    if "y" in categorical_features_assoc:
        categorical_features_assoc.remove("y")

    # On ajoute la cible pour chercher des règles qui concluent vers y = yes
    assoc_cols = categorical_features_assoc + ["y"]

    df_assoc = df[assoc_cols].astype(str).copy()

    # Pour éviter un calcul trop lourd
    if len(df_assoc) > 10000:
        df_assoc = df_assoc.sample(10000, random_state=42)

    print("Nombre d'observations utilisées :", df_assoc.shape[0])
    print("Nombre de variables utilisées :", df_assoc.shape[1])

    # One-hot encoding des modalités
    transactions = pd.get_dummies(df_assoc)

    print("Dimensions de la matrice de transactions :", transactions.shape)

    # Itemsets fréquents
    frequent_itemsets = apriori(
        transactions,
        min_support=0.05,
        use_colnames=True
    )

    print("Nombre d'itemsets fréquents trouvés :", len(frequent_itemsets))

    if len(frequent_itemsets) == 0:
        print("Aucun itemset fréquent trouvé. Essayez de diminuer min_support.")
    else:
        # Règles d'association
        rules = association_rules(
            frequent_itemsets,
            metric="lift",
            min_threshold=1.0
        )

        print("Nombre de règles extraites :", len(rules))

        # On cherche automatiquement la modalité positive de la cible
        target_candidates = [
            col for col in transactions.columns
            if col in ["y_yes", "y_1"]
        ]

        if len(target_candidates) == 0:
            print("Aucune modalité cible positive trouvée.")
            print("Colonnes liées à y :")
            print([col for col in transactions.columns if col.startswith("y_")])
        else:
            target_item = target_candidates[0]
            print("Modalité cible utilisée :", target_item)

            rules_y_yes = rules[
                rules["consequents"].apply(lambda x: target_item in x)
            ].copy()

            rules_y_yes = rules_y_yes.sort_values(
                ["lift", "confidence"],
                ascending=False
            )

            display(
                rules_y_yes[
                    [
                        "antecedents",
                        "consequents",
                        "support",
                        "confidence",
                        "lift",
                        "leverage",
                        "conviction"
                    ]
                ].head(10)
            )

### Questions

1. Quelles règles ont le lift le plus élevé ?
2. Une règle avec lift élevé mais support très faible est-elle utile ?
3. Les règles d'association sont-elles des modèles prédictifs ou des outils descriptifs ?
4. Les règles trouvées sont-elles cohérentes avec les variables importantes de la Random Forest ?


### Réponses aux questions

#### 1. Quelles règles ont le lift le plus élevé ?

Les règles présentant les lifts les plus élevés sont: `contact_cellular`, `default_no` et `loan_no`.

#### 2. Une règle avec lift élevé mais support très faible est-elle utile ?

Pas nécessairement. Un lift élevé indique une association forte, mais si le support est très faible, la règle concerne un nombre limité d'observations et peut être peu robuste. Dans la pratique, une règle est généralement plus intéressante lorsqu'elle combine un lift élevé avec un support suffisant pour représenter une part non négligeable de la population.

#### 3. Les règles d'association sont-elles des modèles prédictifs ou des outils descriptifs ?

Les règles d'association sont avant tout des outils descriptifs. Elles permettent de mettre en évidence des cooccurrences fréquentes entre modalités et d'identifier des profils de clients associés à certains comportements. Contrairement aux modèles de classification étudiés précédemment (SVM, Random Forest, Ridge Logistic, etc.), elles ne sont pas conçues pour optimiser directement la prédiction.

#### 4. Les règles trouvées sont-elles cohérentes avec les variables importantes de la Random Forest ?

Oui. Les règles les plus intéressantes font intervenir des variables comme `contact`, `loan` ou `default`, qui apparaissent également parmi les variables explicatives importantes dans l'analyse Random Forest. Cette cohérence suggère que ces caractéristiques jouent effectivement un rôle dans la distinction entre les clients qui souscrivent et ceux qui ne souscrivent pas. Les deux approches mettent donc en évidence des informations similaires, même si elles reposent sur des méthodes différentes.

## 12. Travail demandé pour le compte rendu

Votre rapport doit contenir :

1. Une introduction au problème de classification bancaire.
2. Une description du jeu de données.
3. Une analyse exploratoire de la variable cible et des variables explicatives.
4. Une explication du prétraitement.
5. Une présentation théorique courte de chaque modèle.
6. Une comparaison des performances sur l'ensemble de test.
7. Une comparaison statistique par validation croisée.
8. Une interprétation des résultats.
9. Une conclusion indiquant le modèle recommandé.



### Rappel : métriques, seuil de décision et objectif métier

Dans un problème de classification binaire, un modèle produit souvent un score
ou une probabilité estimée :

$$
\hat{p}(x)=P(Y=1\mid X=x).
$$

Pour transformer cette probabilité en décision, on choisit un seuil \(s\). Par
défaut, on utilise souvent :

$$
s=0.5.
$$

La règle de décision est alors :

$$
\hat{Y}=1 \quad \text{si} \quad \hat{p}(x)\geq s.
$$

Cependant, ce seuil n'est pas toujours optimal. Si la classe positive est rare
ou si les coûts des erreurs sont asymétriques, il peut être préférable
d'utiliser un seuil plus faible ou plus élevé.

Dans le cas d'une campagne marketing, prédire `yes` signifie que la banque
décide de contacter le client. Un faux positif correspond donc à un contact
inutile, tandis qu'un faux négatif correspond à une opportunité commerciale
manquée.

Le choix de la métrique dépend donc de l'objectif :

- le **rappel** est important si l'on veut détecter le plus grand nombre
  possible de clients intéressés ;
- la **précision** est importante si l'on veut éviter de contacter trop de
  clients non intéressés ;
- le **F1-score** cherche un compromis entre précision et rappel ;
- l'**AUC** mesure la capacité globale du modèle à classer les clients
  intéressés avant les clients non intéressés, indépendamment d'un seuil fixé ;
- une **fonction de gain métier** permet de choisir directement le modèle et
  le seuil les plus rentables.

### Question finale : choix du modèle et décision sous contrainte métier

La banque souhaite utiliser un modèle de classification pour sélectionner les
clients à contacter lors d'une campagne marketing. L'objectif n'est pas
seulement de maximiser une métrique statistique, mais de prendre une décision
rentable et robuste.

On note :

- un **vrai positif** : un client prédit comme intéressé et qui souscrit
  effectivement ;
- un **faux positif** : un client contacté mais qui ne souscrit pas ;
- un **faux négatif** : un client non contacté alors qu'il aurait pu souscrire ;
- un **vrai négatif** : un client non contacté et non intéressé.

On suppose que contacter un client a un coût, tandis qu'une souscription
rapporte un gain à la banque.

Répondez aux questions suivantes.

1. Dans ce contexte, pourquoi l'accuracy peut-elle être une métrique trompeuse,
   en particulier si la classe `yes` est minoritaire ?

2. La banque doit-elle privilégier la précision, le rappel, le F1-score ou
   l'AUC ? Votre réponse doit dépendre explicitement de l'objectif métier :
   maximiser le nombre de souscriptions, limiter le coût des contacts inutiles,
   ou maximiser le profit attendu.

3. Proposez une fonction de coût ou de gain associée à la matrice de confusion.
   Par exemple, si une souscription rapporte $G$ euros et si contacter un
   client coûte $C$ euros, exprimez le gain total en fonction des vrais
   positifs et des faux positifs.

4. Pourquoi le seuil de décision par défaut \(0.5\) n'est-il pas nécessairement
   optimal ? Expliquez comment on pourrait choisir un seuil optimal à partir
   des probabilités prédites par les modèles.

5. Comparez le modèle retenu selon le F1-score, selon l'AUC et selon le gain
   métier proposé. Obtient-on toujours le même meilleur modèle ? Interprétez.

6. En utilisant les intervalles de confiance bootstrap, discutez la stabilité
   du classement des modèles. Peut-on affirmer qu'un modèle est clairement
   meilleur que les autres ?

7. En utilisant les tests de McNemar et la correction de Holm, indiquez si les
   différences observées entre les meilleurs modèles sont statistiquement
   significatives.

8. Concluez en recommandant un modèle et une règle de décision à la banque.
   Votre conclusion doit mentionner :
   
   - la métrique principale retenue ;
   - le rôle du déséquilibre de classes ;
   - le choix éventuel du seuil de classification ;
   - l'incertitude statistique sur les performances ;
   - les limites de l'étude.

### Extension : choix du seuil par maximisation du gain

Supposons que chaque client contacté coûte $C$ euros et qu'une souscription
rapporte $G$ euros. Pour un seuil $s$, on définit le gain :

$$
\operatorname{Gain}(s)
=
G \times TP(s)
-
C \times \left[TP(s)+FP(s)\right].
$$

1. Calculez ce gain pour plusieurs seuils \(s \in [0,1]\).
2. Représentez graphiquement le gain en fonction du seuil.
3. Déterminez le seuil qui maximise le gain.
4. Comparez ce seuil au seuil par défaut \(0.5\).
5. Le modèle qui maximise l'AUC est-il nécessairement celui qui maximise le
   gain métier ? Justifiez.

## Réponses à la question finale : choix du modèle et décision sous contrainte métier

### 1. Pourquoi l'accuracy peut-elle être trompeuse ?

L'accuracy peut être trompeuse lorsque les classes sont déséquilibrées. Dans notre jeu de données, la classe `yes` est minoritaire. Un modèle qui prédirait systématiquement `no` pourrait obtenir une accuracy élevée tout en étant incapable d'identifier les clients susceptibles de souscrire.

L'accuracy ne tient donc pas suffisamment compte des performances sur la classe minoritaire, qui est pourtant celle qui intéresse la banque.

### 2. Quelle métrique privilégier ?

Dans notre étude, le F1-score apparaît comme la métrique la plus adaptée car il tient compte simultanément de la précision et du rappel dans un contexte de déséquilibre de classes.

### 3. Proposition d'une fonction de gain

Si une souscription rapporte \(G\) euros et qu'un contact coûte \(C\) euros, le gain total peut s'écrire :

$$
Gain = G \times TP - C \times (TP + FP).
$$

Chaque vrai positif génère un gain \(G\), tandis que chaque client contacté génère un coût \(C\), qu'il souscrive ou non.

Cette fonction permet de relier directement les performances statistiques à un objectif économique.

### 4. Pourquoi le seuil 0,5 n'est-il pas forcément optimal ?

Le seuil de décision de 0,5 est une convention mais il ne correspond pas nécessairement à la stratégie optimale de la banque.

Dans un problème déséquilibré, diminuer le seuil peut permettre d'augmenter le rappel et donc de détecter davantage de clients susceptibles de souscrire. À l'inverse, augmenter le seuil réduit le nombre de faux positifs mais risque d'augmenter les faux négatifs.

Le seuil optimal peut être déterminé en calculant le gain métier pour plusieurs valeurs de seuil et en sélectionnant celui qui maximise ce gain.

### 5. Comparaison des modèles selon différents critères

Les résultats montrent que le meilleur modèle dépend du critère retenu.

- La Ridge Logistic obtient le meilleur F1-score moyen en validation croisée (≈ 0,58).
- Elle présente également la meilleure AUC moyenne (≈ 0,935).
- Le modèle maximisant le gain métier pourrait être différent selon les valeurs choisies pour \(G\) et \(C\) ainsi que selon le seuil retenu.

Ainsi, le meilleur modèle statistique n'est pas toujours le meilleur modèle économique.

### 6. Stabilité des résultats

Les validations croisées répétées montrent que plusieurs modèles obtiennent des performances relativement proches.

Les distributions des F1-scores indiquent que la Ridge Logistic, le SVM RBF équilibré et le SVM linéaire équilibré appartiennent au groupe des modèles les plus performants.

Il est donc difficile d'affirmer qu'un modèle domine très largement tous les autres sur tous les échantillons possibles.

### 7. Résultats des tests statistiques

Le test global de Friedman rejette l'hypothèse d'égalité parfaite des performances entre les modèles.

Les comparaisons deux à deux montrent que la Ridge Logistic est significativement meilleure que plusieurs modèles moins performants après correction de Holm.

En revanche, les différences entre la Ridge Logistic et les meilleurs SVM équilibrés ne sont pas toujours statistiquement significatives. Les performances de ces modèles doivent donc être considérées comme proches.

### 8. Recommandation finale

Compte tenu du déséquilibre de classes, l'accuracy n'est pas une métrique adaptée pour sélectionner le modèle.

Nous recommandons la Ridge Logistic car elle obtient :
- le meilleur F1-score moyen ;
- la meilleure AUC moyenne ;
- des performances stables en validation croisée.

Dans une application réelle, le seuil de classification devrait être optimisé à partir d'une fonction de gain métier plutôt que fixé arbitrairement à 0,5.

Enfin, les résultats doivent être interprétés avec prudence car certaines différences entre les meilleurs modèles ne sont pas statistiquement significatives. Une analyse complémentaire fondée sur le gain économique réel permettrait de confirmer le choix final.

## Barème détaillé sur 20 points

Le barème suivant tient compte de l'ensemble du projet : prétraitement, variantes de Naive Bayes, modèles de classification, recherche d'hyperparamètres, intervalles de confiance bootstrap, comparaison statistique par McNemar, règles d'association et question finale de décision métier. L'objectif n'est pas seulement d'obtenir le meilleur score, mais de justifier les choix méthodologiques, d'évaluer l'incertitude et de relier les résultats statistiques à une décision bancaire concrète.

| Partie évaluée | Points | Attendus principaux |
|---|---:|---|
| **1. Compréhension du problème et du jeu de données** | **1,5 pts** | Présentation claire de la problématique bancaire, identification de la variable cible, justification du caractère supervisé et binaire du problème, description synthétique des variables disponibles et de l'objectif de prédiction. |
| **2. Analyse exploratoire des données** | **1,5 pts** | Étude de la distribution de la variable cible, mise en évidence du déséquilibre des classes, statistiques descriptives pertinentes, visualisations commentées et premières hypothèses sur les variables utiles. |
| **3. Prétraitement et absence de fuite de données** | **2 pts** | Séparation correcte apprentissage/test, identification des variables numériques et catégorielles, imputation si nécessaire, one-hot encoding des variables catégorielles, standardisation des variables numériques, utilisation justifiée d'un `Pipeline` pour éviter le *data leakage*. |
| **4. Variantes de Naive Bayes** | **1,5 pts** | Comparaison pertinente entre GaussianNB sur variables numériques, CategoricalNB sur variables catégorielles encodées ordinalement et BernoulliNB sur variables one-hot ; explication des hypothèses probabilistes et discussion des performances obtenues. |
| **5. Mise en oeuvre des modèles de classification** | **2 pts** | Entraînement correct de CART, Random Forest, SVM, réseau de neurones et régression logistique Ridge ; code clair, reproductible et cohérent avec les prétraitements. |
| **6. Recherche d'hyperparamètres et prise en compte du déséquilibre** | **2 pts** | Recherche en grille pour le SVM, comparaison de plusieurs noyaux et valeurs de paramètres, comparaison `class_weight=None` et `class_weight="balanced"`, discussion de l'effet du déséquilibre sur la frontière de décision et sur le rappel de la classe minoritaire. |
| **7. Évaluation des performances prédictives** | **1,5 pts** | Calcul et interprétation de l'accuracy, précision, rappel, F1-score, AUC et matrices de confusion ; discussion du choix des métriques dans un contexte de classe minoritaire. |
| **8. Intervalles de confiance bootstrap** | **1,75 pts** | Construction correcte d'intervalles de confiance bootstrap à 95 % pour les métriques, visualisation des intervalles, interprétation de l'incertitude, distinction entre estimation ponctuelle et variabilité d'échantillonnage. |
| **9. Comparaison statistique des classifieurs** | **2 pts** | Utilisation correcte du test de McNemar pour comparer deux classifieurs sur les mêmes observations, construction des tables de contingence, correction de Holm pour comparaisons multiples, interprétation prudente des p-valeurs et des différences significatives. |
| **10. Biais-variance, risque et généralisation** | **1,5 pts** | Explication du lien entre risque empirique et risque réel, compromis biais-variance, courbes d'apprentissage, effet de la complexité des modèles, lien avec les SVM, la marge et les bornes leave-one-out. |
| **11. Règles d'association** | **1 pt** | Extraction et interprétation de règles d'association sur les variables catégorielles, utilisation correcte du support, de la confiance et du lift, discussion du caractère descriptif des règles et de leur lien éventuel avec les variables importantes. |
| **12. Question finale : décision métier, seuil et gain attendu** | **1,5 pts** | Analyse critique du choix entre accuracy, précision, rappel, F1-score et AUC ; proposition d'une fonction de coût ou de gain basée sur la matrice de confusion ; discussion du seuil de décision, du rôle des faux positifs et faux négatifs, et recommandation argumentée pour la banque. |
| **13. Qualité générale du notebook et conclusion** | **0,25 pt** | Notebook structuré, exécutable de bout en bout, commentaires lisibles, graphiques correctement titrés, conclusion synthétique mentionnant le modèle retenu, ses limites et les pistes d'amélioration. |
| **Total** | **20 pts** |  |
